# IFRS S1/S2 Requirements Extraction — Required Hybrid Azure OpenAI Version

## Engineering approach

This notebook keeps the deterministic IFRS extraction pipeline as the auditable baseline, then adds a **required** Azure OpenAI GPT-5.2 hybrid review layer.

The deterministic layer still handles the parts that should remain exact and reproducible: PDF parsing, paragraph reconstruction, official TOC-based body section mapping, clause-marker splitting, traceability, validation and baseline exports.

The hybrid layer is now mandatory. If Azure configuration is missing or invalid, the notebook raises a clear error instead of silently skipping review. By default, GPT-5.2 reviews the highest-impact candidate rows, especially appendix guidance and ambiguous classifications. It can semantically review section mapping, obligation type, generation bucket, evidence tags, banking relevance and split quality. Original rule-based columns are preserved, and hybrid decisions are written to separate `final_*` and `hybrid_*` columns for comparison.

Appendix A / defined terms remains excluded from the report-generation KB by design. It should be extracted separately as a glossary if needed, not mixed with disclosure requirements.


## Configuration

Put `ifrs_s1.pdf` and `ifrs_s2.pdf` in the same folder as this notebook. In this environment the notebook also falls back to `/mnt/data` automatically.

In [1]:
from pathlib import Path
import re, json, math
from collections import defaultdict, Counter
import fitz
import pandas as pd
from IPython.display import display, Markdown

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'ifrs_s1.pdf').exists() and Path('gen_data/IFRS/ifrs_s1.pdf').exists():
    BASE_DIR = Path('gen_data/IFRS/')
PDF_SOURCES = {
    'IFRS S1': BASE_DIR / 'ifrs_s1.pdf',
    'IFRS S2': BASE_DIR / 'ifrs_s2.pdf',
}
TARGET_REPORT_SECTIONS = [
    'General Requirements', 'Governance', 'Strategy', 'Risk Management', 'Metrics and Targets'
]
OUTPUT_DIR = BASE_DIR / 'ifrs_requirements_kb_outputs_hybrid'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARA_ID_RE = re.compile(r'^\s*(?P<id>[A-Z]?\d+[A-Z]?)(?:\s+|$)(?P<rest>.*)$')
BODY_PARA_ID_RE = re.compile(r'^\s*(?P<id>\d+[A-Z]?)(?:\s+|$)(?P<rest>.*)$')
LIST_MARKER_RE = re.compile(r'(?<!\w)(\([a-z]\)|\((?:i|ii|iii|iv|v|vi|vii|viii|ix|x)\)|\([1-9]\))\s+', re.I)
KNOWN_HEADER_RE = re.compile(
    r'^(IFRS SUSTAINABILITY DISCLOSURE STANDARDS|IFRS S1 GENERAL REQUIREMENTS|RELATED FINANCIAL INFORMATION|IFRS S2 CLIMATE-RELATED DISCLOSURES|© IFRS Foundation|\d+\s+© IFRS Foundation|© IFRS Foundation\s+\d+)$',
    re.I,
)

GENERIC_HEADING_STARTS = {
    'Objective','Scope','Conceptual foundations','Fair presentation','Materiality','Reporting entity','Connected information',
    'Core content','Governance','Strategy','Risk management','Metrics and targets','General requirements','Sources of guidance',
    'Identifying sustainability-related risks and opportunities','Identifying applicable disclosure requirements',
    'Disclosure of information about sources of guidance','Location of disclosures','Timing of reporting','Comparative information',
    'Statement of compliance','Judgements, uncertainties and errors','Judgements','Measurement uncertainty','Errors',
    'Sustainability-related risks and opportunities','Business model and value chain','Strategy and decision-making',
    'Financial position, financial performance and cash flows','Resilience',
    'Climate-related risks and opportunities','Climate resilience','Climate-related metrics','Climate-related targets',
    'Appendix A','Appendix B','Appendix C','Appendix D','Appendix E','Defined terms','Application guidance',
    'Effective date and transition','Effective date','Transition','Transition for Amendments to Greenhouse Gas','Financed emissions',
    'Asset management','Commercial banking','Insurance','Cross-industry metric categories','Greenhouse gases',
    'Greenhouse gas emissions','Scope 2 greenhouse gas emissions','Scope 3 greenhouse gas emissions','Scope 3 measurement framework',
    'Measurement approach, inputs and assumptions','Carbon credits','Gross and net greenhouse gas emissions targets',
    'Characteristics of a climate-related target','Greenhouse gas emissions targets','Selecting inputs','Making analytical choices',
    'Additional considerations','Exposure to climate-related risks and opportunities','Skills, capabilities and resources available',
    'Determining the appropriate approach','Reasonable and supportable information','Assessing the circumstances',
    'Aggregation and disaggregation','Interaction with law or regulation','Commercially sensitive information',
    'Information included by cross-reference','Interim reporting','Metrics','Potential reporting period errors discovered in that period are corrected before'
}


def normalize_ws(text):
    if text is None:
        return ''
    text = str(text)
    # PDF artefacts and non-standard hyphenation controls
    text = text.replace('\u00ad', '').replace('\ufffe', '').replace('\ufeff', '')
    text = text.replace('\u2011', '-').replace('\u2010', '-').replace('\u2012', '-')
    # restore words broken as "climate- related" after line joining
    text = re.sub(r'(\w)-\s+(?=[A-Za-z])', r'\1-', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace(' ,', ',').replace(' .', '.').replace(' ;', ';').replace(' :', ':')
    text = text.replace('( ', '(').replace(' )', ')')
    return text.strip()




def remove_trailing_footnote_markers(text):
    """Remove PDF footnote markers that remain attached to the final punctuation.

    Example: "... entity.1" -> "... entity."
    The rule only applies at the very end of a reconstructed paragraph/requirement,
    so it does not affect valid terms such as Scope 1 or paragraph references.
    """
    text = normalize_ws(text)
    text = re.sub(r'(?<=[.!?])\d{1,2}$', '', text)
    return normalize_ws(text)


def clean_extracted_text(text):
    """Final text cleanup after paragraph assembly or requirement splitting."""
    text = normalize_ws(text)
    text = remove_trailing_footnote_markers(text)
    # Normalise repeated dashes caused by nested context assembly.
    text = re.sub(r'\s+—\s+—\s+', ' — ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def clean_requirement_text_for_generation(text):
    """Readable requirement version for generation agents.

    Keeps paragraph context, but removes PDF artefacts and repeated separators.
    Full traceability remains available through source_paragraph_text.
    """
    text = clean_extracted_text(text)
    text = re.sub(r'\s+—\s+', ' — ', text)
    text = re.sub(r'\s*;\s*$', '', text)
    return text.strip()


def is_objective_context(text):
    return normalize_ws(text).lower().startswith('the objective of')


def classify_generation_bucket(obligation_type, is_leaf_requirement, text):
    """Separate rows used for generation from context/guidance rows."""
    if is_objective_context(text):
        return 'supporting_objective'
    if not bool(is_leaf_requirement):
        return 'container_context'
    if obligation_type in {'mandatory', 'prohibition'}:
        return 'must_disclose_leaf'
    if obligation_type == 'relief_or_optional':
        return 'optional_relief_leaf'
    return 'supporting_guidance_leaf'


def build_generation_ready_df(requirements_df):
    """Create a smaller KB for report-generation agents.

    It keeps only actionable mandatory leaf disclosure rows and exposes the
    clean text field first, while preserving source paragraph traceability.
    """
    gen = requirements_df[
        (requirements_df['is_leaf_requirement'] == True) &
        (requirements_df['mandatory'] == True) &
        (requirements_df['generation_bucket'] == 'must_disclose_leaf')
    ].copy()
    preferred_cols = [
        'requirement_id', 'standard', 'paragraph_id', 'page', 'report_section',
        'official_section_heading', 'nearest_pdf_heading', 'clean_requirement_text',
        'source_paragraph_text', 'clause_path', 'obligation_type', 'mandatory',
        'evidence_tags', 'banking_relevance', 'paragraph_quality_score',
        'requirement_quality_score'
    ]
    return gen[[c for c in preferred_cols if c in gen.columns]].reset_index(drop=True)

def paragraph_family(pid):
    m = re.match(r'^([A-Z]?)(\d+)([A-Z]?)$', str(pid))
    if not m:
        return ('', None, '')
    return m.group(1), int(m.group(2)), m.group(3)


def paragraph_sort_key(pid):
    prefix, number, suffix = paragraph_family(pid)
    prefix_rank = {'': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}.get(prefix, 99)
    suffix_rank = ord(suffix) - ord('A') + 1 if suffix else 0
    return (prefix_rank, number if number is not None else 10**9, suffix_rank)


def clean_line_text(text):
    return normalize_ws(text)


def get_ordered_lines(page):
    """Extract sorted visual lines with coordinates.

    Uses PyMuPDF's layout dictionary instead of OCR. This keeps paragraph numbers,
    list markers and continuation text in reading order while allowing footer/header removal.
    """
    lines = []
    for block in page.get_text('dict', sort=True).get('blocks', []):
        if block.get('type') != 0:
            continue
        for line in block.get('lines', []):
            parts = []
            for span in line.get('spans', []):
                parts.append(span.get('text', ''))
            text = clean_line_text(''.join(parts))
            if not text:
                continue
            x0, y0, x1, y1 = line.get('bbox', [0, 0, 0, 0])
            lines.append({'text': text, 'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1})
    # Same-baseline fragments can appear as separate line objects. Group by y first,
    # then sort fragments by x inside each baseline. This avoids losing lines where the
    # paragraph number is slightly lower than its text baseline (for example IFRS S1.20).
    raw = sorted(lines, key=lambda r: (r['y0'], r['x0']))
    groups = []
    for line in raw:
        placed = False
        for group in groups:
            if abs(group['y'] - line['y0']) < 1.7:
                group['items'].append(line)
                group['y'] = sum(item['y0'] for item in group['items']) / len(group['items'])
                placed = True
                break
        if not placed:
            groups.append({'y': line['y0'], 'items': [line]})
    merged = []
    for group in sorted(groups, key=lambda g: g['y']):
        items = sorted(group['items'], key=lambda r: r['x0'])
        text = ''
        for item in items:
            curr = item['text']
            if not text:
                text = curr
            else:
                sep = '' if (text.endswith('-') or curr.startswith((',', '.', ';', ':', ')'))) else ' '
                text = normalize_ws(text + sep + curr)
        merged.append({
            'text': normalize_ws(text),
            'x0': min(i['x0'] for i in items),
            'y0': min(i['y0'] for i in items),
            'x1': max(i['x1'] for i in items),
            'y1': max(i['y1'] for i in items),
        })
    return merged


def is_noise_line(line):
    t = normalize_ws(line['text'])
    y = line['y0']
    x = line['x0']
    if not t:
        return True
    if y < 115:
        return True
    if y > 724:
        return True
    if KNOWN_HEADER_RE.match(t):
        return True
    # Page number alone near footer
    if y > 700 and re.fullmatch(r'\d+', t):
        return True
    # Footnotes in these PDFs, without removing real body list items at the bottom.
    footnote_starts = (
        'Throughout this Standard,', 'the same meaning.', 'This application guidance',
        'documents published by the Task Force', 'including Technical Supplement:',
        'Opportunities (2017) and Guidance', 'and Opportunities (2017) and Guidance'
    )
    if y > 650 and (
        t.startswith(footnote_starts)
        or re.match(r'^\d+\s+(Throughout this Standard|This application guidance)', t)
        or (re.fullmatch(r'\d+', t) and x < 170)
    ):
        return True
    return False


def line_starts_paragraph(line_text, body_started=True):
    m = PARA_ID_RE.match(line_text)
    if not m:
        return None, None
    pid = m.group('id')
    rest = normalize_ws(m.group('rest'))
    prefix, num, suffix = paragraph_family(pid)
    if num is None:
        return None, None
    # Accept body paragraphs and appendix paragraph IDs. Avoid ordinary sentence-leading numbers.
    if prefix == '' or prefix in {'B', 'C', 'D', 'E'}:
        return pid, rest
    return None, None


def is_probable_heading(text, known_titles=None, strict=False):
    t = normalize_ws(text)
    if not t:
        return False
    if re.fullmatch(r'\([a-zivxlcdm0-9]+\)', t, re.I):
        return False
    if re.match(r'^\([a-zivxlcdm0-9]+\)\s+', t, re.I):
        return False
    if known_titles and t.lower() in {h.lower() for h in known_titles}:
        return True
    if t in GENERIC_HEADING_STARTS:
        return True
    if re.match(r'^Appendix\s+[A-Z]$', t):
        return True
    if len(t) <= 115 and re.search(r'\(paragraphs?\s+[A-Z]?\d+', t, re.I):
        return True
    if strict:
        return False
    if len(t) <= 90 and not re.search(r'[.;:]$', t):
        # A short line with mostly title case words is usually a heading when we are not
        # inside a paragraph. Inside a paragraph this heuristic is intentionally disabled
        # to avoid dropping standard names such as "Greenhouse Gas Protocol: A Corporate Accounting and".
        words = re.findall(r'[A-Za-z]+', t)
        if words:
            if t.upper() == t and len(words) <= 8:
                return True
            titleish = sum(1 for w in words if w[:1].isupper()) >= max(1, math.ceil(len(words) * 0.55))
            # Avoid classifying normal lowercase continuation lines as headings.
            if titleish and not re.search(r'\b(shall|entity|users|risks|opportunities|information)\b', t, re.I):
                return True
    return False


def find_body_start_page(pdf_path):
    doc = fitz.open(pdf_path)
    for i, page in enumerate(doc):
        lines = [l for l in get_ordered_lines(page) if not is_noise_line(l)]
        texts = [l['text'] for l in lines]
        joined = ' '.join(texts)
        has_objective = any(normalize_ws(t).lower() == 'objective' for t in texts)
        has_para_1 = any(line_starts_paragraph(t)[0] == '1' for t in texts)
        # Avoid contents pages: require prose from objective paragraph.
        if has_objective and has_para_1 and re.search(r'objective of IFRS S[12]', joined, re.I):
            return i
    # Fallback: first page after contents with paragraph 1
    for i, page in enumerate(doc):
        lines = [l for l in get_ordered_lines(page) if not is_noise_line(l)]
        if any(line_starts_paragraph(l['text'])[0] == '1' for l in lines):
            return i
    return 0


def extract_toc_entries(pdf_path, standard):
    doc = fitz.open(pdf_path)
    entries = []
    found = False
    for page_index in range(min(8, len(doc))):
        text = doc[page_index].get_text('text', sort=True)
        lines = [normalize_ws(x) for x in text.splitlines() if normalize_ws(x)]
        page_join = ' '.join(lines).upper()
        if 'CONTENTS' not in page_join and not found:
            continue
        found = True
        for line in lines:
            # Examples: "Governance 26", "GENERAL REQUIREMENTS 54".
            m = re.match(r'^(?P<title>.+?)\s+(?P<start>[A-Z]?\d+[A-Z]?)$', line)
            if not m:
                continue
            title = normalize_ws(m.group('title'))
            start = m.group('start')
            if title.lower() in {'from paragraph', 'continued...', '...continued'}:
                continue
            if title.upper().startswith(('IFRS ', 'S1 ', 'S2 ', 'APPROVAL', 'FOR ', 'ILLUSTRATIVE', 'BASIS')):
                continue
            if re.search(r'©|Foundation', title):
                continue
            prefix, num, suffix = paragraph_family(start)
            if num is None:
                continue
            entries.append({
                'standard': standard,
                'toc_title': title,
                'start_paragraph': start,
                'toc_page': page_index + 1,
                'is_uppercase_heading': title.upper() == title,
            })
        if found and entries:
            # These PDFs have the full relevant TOC on one page.
            break
    # Compute paragraph end for body entries only.
    body_entries = [e for e in entries if paragraph_family(e['start_paragraph'])[0] == '']
    for i, e in enumerate(body_entries):
        curr = paragraph_family(e['start_paragraph'])[1]
        next_nums = [paragraph_family(n['start_paragraph'])[1] for n in body_entries[i+1:]
                     if paragraph_family(n['start_paragraph'])[1] and paragraph_family(n['start_paragraph'])[1] > curr]
        e['end_paragraph'] = (min(next_nums) - 1) if next_nums else None
    body_lookup = {(e['toc_title'], e['start_paragraph']): e.get('end_paragraph') for e in body_entries}
    for e in entries:
        e['end_paragraph'] = body_lookup.get((e['toc_title'], e['start_paragraph']))
    return entries


def append_part(parts, line):
    line = normalize_ws(line)
    if not line:
        return
    if parts and re.search(r'\w-$', parts[-1]):
        parts[-1] = normalize_ws(parts[-1] + line)
    else:
        parts.append(line)


def extract_paragraphs(pdf_path, standard, known_toc_titles=None):
    doc = fitz.open(pdf_path)
    body_start = find_body_start_page(pdf_path)
    rows = []
    current = None
    current_heading = None
    current_appendix = None
    current_appendix_title = None

    def flush_current():
        nonlocal current
        if current:
            text = clean_extracted_text(' '.join(current['parts']))
            current['text'] = text
            del current['parts']
            rows.append(current)
            current = None

    for page_index, page in enumerate(doc):
        if page_index < body_start:
            continue
        lines = [l for l in get_ordered_lines(page) if not is_noise_line(l)]
        for line in lines:
            txt = normalize_ws(line['text'])
            if not txt:
                continue
            # Appendix tracking before paragraph starts.
            if re.match(r'^Appendix\s+[A-Z]$', txt):
                flush_current()
                current_appendix = txt.split()[-1]
                current_appendix_title = txt
                current_heading = txt
                continue
            # The appendix title and intro sentence are not numbered requirements.
            if current_appendix and txt in {'Application guidance', 'Defined terms', 'Effective date and transition'}:
                current_appendix_title = txt
                current_heading = txt
                continue
            # Detect new paragraph. A paragraph ID can be in a standalone margin line.
            pid, rest = line_starts_paragraph(txt)
            if pid:
                prefix, num, suffix = paragraph_family(pid)
                # Avoid accidental page numbers or footnotes.
                if line['x0'] <= 145 and not (line['y0'] > 650 and prefix == '' and num in {1, 2} and not rest):
                    flush_current()
                    current = {
                        'standard': standard,
                        'paragraph_id': pid,
                        'page': page_index + 1,
                        'appendix': current_appendix,
                        'appendix_title': current_appendix_title,
                        'nearest_heading': current_heading,
                        'parts': []
                    }
                    if rest:
                        append_part(current['parts'], rest)
                    continue
            # Heading lines should update context and must not pollute paragraph text.
            if is_probable_heading(txt, known_toc_titles, strict=(current is not None)):
                current_heading = txt
                continue
            # Some appendix intro prose is not paragraph text and occurs before B/C/D/E numbering.
            if current is None:
                continue
            append_part(current['parts'], txt)
    flush_current()
    return rows


def classify_toc_title(title, standard):
    t = normalize_ws(title).lower()
    if t == 'governance':
        return 'Governance'
    if t == 'strategy':
        return 'Strategy'
    if t == 'risk management':
        return 'Risk Management'
    if t == 'metrics and targets':
        return 'Metrics and Targets'
    general_keywords = [
        'objective', 'scope', 'conceptual', 'fair presentation', 'materiality', 'reporting entity',
        'connected information', 'core content', 'general requirements', 'sources of guidance',
        'location of disclosures', 'timing of reporting', 'comparative information', 'statement of compliance',
        'judgements', 'measurement uncertainty', 'errors'
    ]
    if any(k in t for k in general_keywords):
        return 'General Requirements'
    # Subheadings under target sections inherit from parent by range. These direct titles also map semantically.
    if any(k in t for k in ['business model', 'decision-making', 'financial position', 'resilience', 'climate-related risks']):
        return 'Strategy'
    if any(k in t for k in ['climate-related metrics', 'climate-related targets', 'greenhouse gas', 'cross-industry metric']):
        return 'Metrics and Targets'
    return None


def build_body_section_map(toc_entries, paragraphs_df):
    rows = []
    body_only = paragraphs_df[(paragraphs_df['paragraph_prefix'] == '') & paragraphs_df['appendix'].isna()]
    existing_nums = {std: sorted(g['paragraph_number'].dropna().astype(int).unique()) for std, g in body_only.groupby('standard')}
    for e in toc_entries:
        section = classify_toc_title(e['toc_title'], e['standard'])
        if not section:
            continue
        prefix, start, suffix = paragraph_family(e['start_paragraph'])
        if prefix or start is None:
            continue
        end = e.get('end_paragraph')
        if end is None:
            nums = [n for n in existing_nums.get(e['standard'], []) if n >= start]
            end = max(nums) if nums else start
        rows.append({**e, 'report_section': section, 'start_number': start, 'end_number': int(end)})
    # If multiple headings start at the same paragraph, keep the one with the smallest range / most specific section.
    by_key = defaultdict(list)
    for r in rows:
        by_key[(r['standard'], r['start_number'], r['report_section'])].append(r)
    unique = []
    for key, vals in by_key.items():
        unique.append(sorted(vals, key=lambda r: (r['end_number'] - r['start_number'], r['toc_title'].isupper()))[0])
    return sorted(unique, key=lambda r: (r['standard'], r['start_number'], r['end_number']))


def assign_body_sections(paragraphs_df, section_ranges):
    df = paragraphs_df.copy()
    df['report_section'] = None
    df['official_section_heading'] = None
    df['mapping_method'] = None
    ranges = sorted(section_ranges, key=lambda r: (r['standard'], r['end_number'] - r['start_number']))
    for idx, row in df.iterrows():
        if row['paragraph_prefix'] != '' or pd.notna(row.get('appendix')):
            continue
        num = row['paragraph_number']
        matches = [r for r in ranges if r['standard'] == row['standard'] and r['start_number'] <= num <= r['end_number']]
        if not matches:
            continue
        matches = sorted(matches, key=lambda r: (r['end_number'] - r['start_number'], r['report_section'] == 'General Requirements'))
        m = matches[0]
        df.at[idx, 'report_section'] = m['report_section']
        df.at[idx, 'official_section_heading'] = m['toc_title']
        df.at[idx, 'mapping_method'] = 'auto_toc_body_range'
    return df


def extract_referenced_paragraphs(text):
    refs = []
    # Captures paragraph 22, paragraphs B1–B18, paragraph 29(a), see paragraphs 33–36.
    for m in re.finditer(r'paragraphs?\s+([A-Z]?\d+[A-Z]?)(?:\s*[–-]\s*([A-Z]?\d+[A-Z]?))?', str(text), re.I):
        refs.append(m.group(1))
        if m.group(2):
            refs.append(m.group(2))
    return refs


def map_appendix_sections(df):
    df = df.copy()
    body_map = {(r.standard, r.paragraph_id): r.report_section for r in df.itertuples() if r.paragraph_prefix == '' and pd.notna(r.report_section)}
    # Map appendix ranges by direct references in heading/text or by strong heading keywords.
    for idx, row in df[df['paragraph_prefix'] != ''].iterrows():
        std = row['standard']
        mapped = None
        for source in [row.get('nearest_heading', ''), row.get('text', '')]:
            for ref in extract_referenced_paragraphs(source):
                if (std, ref) in body_map:
                    mapped = body_map[(std, ref)]
                    break
            if mapped:
                break
        if not mapped:
            combined = normalize_ws(str(row.get('nearest_heading', '')) + ' ' + str(row.get('text', ''))[:400]).lower()
            if any(k in combined for k in ['greenhouse gas', 'scope 3', 'scope 2', 'scope 1', 'financed emissions', 'commercial banking', 'asset management', 'insurance', 'gross exposure', 'carbon credit', 'climate-related targets', 'cross-industry metric', 'metrics']):
                mapped = 'Metrics and Targets'
            elif any(k in combined for k in ['climate resilience', 'scenario analysis', 'transition plan', 'business model']):
                mapped = 'Strategy'
            elif any(k in combined for k in ['risk management', 'identify, assess, prioritise', 'prioritise and monitor']):
                mapped = 'Risk Management'
            elif any(k in combined for k in ['governance', 'board', 'management\'s role']):
                mapped = 'Governance'
            elif any(k in combined for k in ['materiality', 'connected information', 'sources of guidance', 'qualitative characteristics', 'reporting entity', 'cross-reference', 'comparative information', 'errors', 'commercially sensitive', 'law or regulation', 'general purpose financial']):
                mapped = 'General Requirements'
        if mapped:
            df.at[idx, 'report_section'] = mapped
            df.at[idx, 'official_section_heading'] = row.get('nearest_heading')
            df.at[idx, 'mapping_method'] = 'auto_appendix_reference_or_heading'
    return df


def marker_level(marker):
    inner = marker.strip('()').lower()
    if inner.isdigit():
        return 3
    if inner in {'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x'}:
        return 2
    return 1


def split_requirement_clauses(paragraph_text):
    text = clean_extracted_text(paragraph_text)
    matches = list(LIST_MARKER_RE.finditer(text))
    if len(matches) < 1:
        return [{
            'clause_marker': None,
            'clause_level': 0,
            'clause_path': '',
            'parent_context': '',
            'requirement_text': text,
            'is_leaf_requirement': True,
        }]
    preamble = text[:matches[0].start()].strip(' :;')
    raw = []
    for i, m in enumerate(matches):
        end = matches[i+1].start() if i + 1 < len(matches) else len(text)
        marker = m.group(1)
        body = normalize_ws(text[m.end():end].strip(' ;'))
        # Remove list-joining conjunctions that belong to the typography, not to the requirement itself.
        body = re.sub(r'(?:;|,)?\s+(?:and|or)$', '', body, flags=re.I)
        lvl = marker_level(marker)
        raw.append({'marker': marker, 'level': lvl, 'body': body})
    # Identify parent markers: a row is a container if the next marker is deeper.
    for i, item in enumerate(raw):
        item['is_leaf_requirement'] = not (i + 1 < len(raw) and raw[i+1]['level'] > item['level'])
    stack = {}
    clauses = []
    for item in raw:
        lvl = item['level']
        # Remove deeper context when returning to a higher level.
        for k in list(stack):
            if k >= lvl:
                del stack[k]
        ancestors = [stack[k] for k in sorted(stack) if k < lvl]
        context_parts = []
        if preamble:
            context_parts.append(preamble)
        for anc in ancestors:
            context_parts.append(f"{anc['marker']} {anc['body']}")
        context = normalize_ws(' — '.join(context_parts))
        req_text = normalize_ws(' — '.join([p for p in [context, f"{item['marker']} {item['body']}"] if p]))
        path = '>'.join([anc['marker'] for anc in ancestors] + [item['marker']])
        clauses.append({
            'clause_marker': item['marker'],
            'clause_level': lvl,
            'clause_path': path,
            'parent_context': context,
            'requirement_text': req_text,
            'is_leaf_requirement': item['is_leaf_requirement'],
        })
        stack[lvl] = item
    return clauses

KEYWORD_TAGS = {
    'governance_body': ['governance body', 'board', 'committee', 'individual(s) responsible'],
    'management_role': ['management’s role', 'management-level', 'management uses controls'],
    'strategy_decision_making': ['strategy and decision-making', 'responded to', 'plans to respond', 'transition plan', 'resource allocation'],
    'business_model_value_chain': ['business model', 'value chain'],
    'financial_effects': ['financial position', 'financial performance', 'cash flows', 'financial effects', 'carrying amounts'],
    'scenario_analysis': ['scenario analysis', 'scenarios', 'climate resilience'],
    'risk_process': ['identify, assess, prioritise', 'risk management process', 'monitor climate-related risks', 'monitor sustainability-related risks', 'overall risk management'],
    'metrics': ['metrics', 'metric', 'cross-industry metric', 'industry-based metric'],
    'targets': ['targets', 'target', 'milestones', 'base period'],
    'ghg_emissions': ['greenhouse gas', 'ghg', 'co2 equivalent', 'scope 1', 'scope 2', 'scope 3'],
    'scope_1': ['scope 1'],
    'scope_2': ['scope 2'],
    'scope_3': ['scope 3'],
    'financed_emissions': ['financed emissions', 'category 15'],
    'commercial_banking': ['commercial banking', 'loans', 'project finance', 'undrawn loan commitments', 'gross exposure'],
    'asset_management': ['asset management', 'assets under management', 'aum'],
    'insurance': ['insurance'],
    'carbon_credits': ['carbon credit', 'carbon credits'],
    'remuneration': ['remuneration'],
    'materiality': ['material ', 'materiality', 'materially', 'obscur'],
    'connected_information': ['connected information', 'connections between', 'cross-reference', 'financial statements'],
    'source_guidance': ['sources of guidance', 'sasb', 'industry-based guidance', 'standard-setting bodies'],
}

def infer_tags(text):
    tl = normalize_ws(text).lower()
    return [tag for tag, kws in KEYWORD_TAGS.items() if any(k.lower() in tl for k in kws)]


def infer_obligation(text):
    tl = clean_extracted_text(text).lower()
    if tl.startswith('the objective of'):
        return 'context_objective'
    # "may" can appear inside examples; check explicit optional phrases first.
    if 'shall not' in tl or 'is prohibited' in tl or 'prohibited from' in tl:
        return 'prohibition'
    if any(k in tl for k in ['need not', 'is permitted', 'are permitted', 'may disclose', 'may apply', 'may refer', 'is not required', 'not required to', 'relief']):
        return 'relief_or_optional'
    if any(k in tl for k in ['shall ', 'shall:', 'shall—', 'is required to', 'are required to', 'requires an entity', 'required by']):
        return 'mandatory'
    return 'guidance'


def infer_banking_relevance(standard, section, text, tags):
    if any(t in tags for t in ['financed_emissions', 'commercial_banking', 'asset_management', 'insurance']):
        return 'high'
    if standard == 'IFRS S2' and section in ['Strategy', 'Risk Management', 'Metrics and Targets']:
        return 'medium'
    if any(t in tags for t in ['financial_effects', 'scope_3', 'ghg_emissions']):
        return 'medium'
    return 'general'


def make_label(text):
    t = re.sub(r'^\([a-zivxlcdm0-9]+\)\s+', '', normalize_ws(text), flags=re.I)
    t = re.sub(r'^(An entity shall disclose|The entity shall disclose|Specifically, the entity shall disclose|To achieve this objective, an entity shall disclose)\s+', '', t, flags=re.I)
    words = t.split()
    return ' '.join(words[:16]) + ('…' if len(words) > 16 else '')


def paragraph_quality(text, known_headings=None):
    text = clean_extracted_text(text)
    issues = []
    if len(text) < 25:
        issues.append('very_short')
    if re.search(r'\b(the|and|or|of|to|with|about|including|considering|would not)$', text, re.I):
        issues.append('dangling_ending')
    if known_headings:
        heading_set = {h.lower() for h in known_headings}
        # Flag if text ends with a heading label appended to paragraph text.
        for h in heading_set:
            if len(h) > 5 and text.lower().endswith(' ' + h):
                issues.append('heading_leakage')
                break
    markers = [m.group(1).strip('()').lower() for m in LIST_MARKER_RE.finditer(text)]
    if {'v', 'vi'}.intersection(markers) and 'iv' not in markers:
        issues.append('missing_roman_iv')
    if 'iii' in markers and 'ii' not in markers:
        issues.append('missing_roman_ii')
    score = max(0.0, 1.0 - 0.2 * len(set(issues)))
    return score, sorted(set(issues))


def clause_quality(text, is_leaf=True):
    text = clean_extracted_text(text)
    issues = []
    if len(text) < 20:
        issues.append('very_short')
    if is_leaf and re.search(r'\b(the|and|or|of|to|with|about|including|considering|would not)$', text, re.I):
        if not re.search(r'committed to$', text, re.I):
            issues.append('dangling_ending')
    score = max(0.0, 1.0 - 0.25 * len(set(issues)))
    return score, sorted(set(issues))


def build_requirements_kb():
    toc_entries = []
    for std, path in PDF_SOURCES.items():
        toc_entries.extend(extract_toc_entries(path, std))
    known_titles = [e['toc_title'] for e in toc_entries] + list(GENERIC_HEADING_STARTS)
    paragraphs = []
    for std, path in PDF_SOURCES.items():
        paragraphs.extend(extract_paragraphs(path, std, known_titles))
    paragraphs_df = pd.DataFrame(paragraphs)
    fam = paragraphs_df['paragraph_id'].apply(paragraph_family)
    paragraphs_df['paragraph_prefix'] = [x[0] for x in fam]
    paragraphs_df['paragraph_number'] = [x[1] for x in fam]
    paragraphs_df['paragraph_suffix'] = [x[2] for x in fam]
    paragraphs_df['paragraph_sort_key'] = paragraphs_df['paragraph_id'].apply(paragraph_sort_key)
    paragraphs_df = paragraphs_df.sort_values(['standard', 'paragraph_sort_key']).drop(columns=['paragraph_sort_key']).reset_index(drop=True)
    section_ranges = build_body_section_map(toc_entries, paragraphs_df)
    paragraphs_df = assign_body_sections(paragraphs_df, section_ranges)
    paragraphs_df = map_appendix_sections(paragraphs_df)
    # Keep only target sections and exclude definitions/effective-date transition appendices.
    selected = paragraphs_df[paragraphs_df['report_section'].isin(TARGET_REPORT_SECTIONS)].copy()
    selected = selected[~((selected['standard'] == 'IFRS S1') & (selected['appendix'].isin(['A', 'E'])))]
    selected = selected[~((selected['standard'] == 'IFRS S2') & (selected['appendix'].isin(['A', 'C'])))]
    selected = selected[selected['text'].str.len() >= 25].copy()
    quality_results = selected['text'].apply(lambda x: paragraph_quality(x, known_titles))
    selected['paragraph_quality_score'] = [q[0] for q in quality_results]
    selected['paragraph_quality_issues'] = [q[1] for q in quality_results]

    req_rows = []
    for row in selected.itertuples(index=False):
        clauses = split_requirement_clauses(row.text)
        for clause_index, clause in enumerate(clauses, start=1):
            req_text = clean_extracted_text(clause['requirement_text'])
            clean_req_text = clean_requirement_text_for_generation(req_text)
            tags = infer_tags(' '.join([clean_req_text, str(row.nearest_heading), str(row.official_section_heading)]))
            obligation = infer_obligation(req_text)
            req_quality, req_issues = clause_quality(req_text, clause['is_leaf_requirement'])
            req_rows.append({
                'requirement_id': f"{row.standard.replace(' ', '_')}_{row.paragraph_id}_C{clause_index:02d}",
                'standard': row.standard,
                'paragraph_id': row.paragraph_id,
                'page': int(row.page),
                'report_section': row.report_section,
                'official_section_heading': row.official_section_heading,
                'nearest_pdf_heading': row.nearest_heading,
                'mapping_method': row.mapping_method,
                'appendix': row.appendix,
                'clause_index': clause_index,
                'clause_marker': clause['clause_marker'],
                'clause_level': clause['clause_level'],
                'clause_path': clause['clause_path'],
                'is_leaf_requirement': bool(clause['is_leaf_requirement']),
                'requirement_label': make_label(req_text),
                'requirement_text': req_text,
                'clean_requirement_text': clean_req_text,
                'parent_context': clean_extracted_text(clause['parent_context']),
                'source_paragraph_text': row.text,
                'obligation_type': obligation,
                'mandatory': obligation in {'mandatory', 'prohibition'},
                'generation_bucket': classify_generation_bucket(obligation, clause['is_leaf_requirement'], clean_req_text),
                'evidence_tags': tags,
                'banking_relevance': infer_banking_relevance(row.standard, row.report_section, req_text, tags),
                'paragraph_quality_score': row.paragraph_quality_score,
                'paragraph_quality_issues': row.paragraph_quality_issues,
                'requirement_quality_score': req_quality,
                'requirement_quality_issues': req_issues,
            })
    requirements_df = pd.DataFrame(req_rows)
    return toc_entries, paragraphs_df, selected, requirements_df, section_ranges


def validate_outputs(paragraphs_df, selected_df, requirements_df, section_ranges):
    rows = []
    def add(check, passed, details):
        rows.append({'check': check, 'passed': bool(passed), 'details': details})
    actual_sections = sorted(requirements_df['report_section'].dropna().unique().tolist()) if len(requirements_df) else []
    add('Only target report sections are present', set(actual_sections).issubset(set(TARGET_REPORT_SECTIONS)), f'actual={actual_sections}')
    add('Every requirement has paragraph traceability', requirements_df[['standard','paragraph_id','page','source_paragraph_text']].notna().all().all(), str(requirements_df[['standard','paragraph_id','page','source_paragraph_text']].isna().sum().to_dict()))
    add('Requirement IDs are unique', requirements_df['requirement_id'].is_unique, f'duplicate_ids={requirements_df.requirement_id.duplicated().sum()}')
    bad_text = requirements_df[requirements_df['requirement_text'].str.len() < 20]
    add('No very short requirement rows', len(bad_text) == 0, f'rows={len(bad_text)}')
    quality_bad = selected_df[selected_df['paragraph_quality_score'] < 0.8]
    add('Paragraph reconstruction quality passed', len(quality_bad) == 0, f'low_quality_rows={len(quality_bad)}')
    required_anchors = [
        ('IFRS S1','20'), ('IFRS S1','23'), ('IFRS S1','25'), ('IFRS S1','29'), ('IFRS S1','35'), ('IFRS S1','44'), ('IFRS S1','46'), ('IFRS S1','51'),
        ('IFRS S2','10'), ('IFRS S2','14'), ('IFRS S2','16'), ('IFRS S2','22'), ('IFRS S2','25'), ('IFRS S2','29'), ('IFRS S2','29A'), ('IFRS S2','29B'), ('IFRS S2','29C'), ('IFRS S2','33'), ('IFRS S2','36'),
        ('IFRS S2','B58'), ('IFRS S2','B59'), ('IFRS S2','B62'), ('IFRS S2','B62A')
    ]
    present = set(zip(selected_df['standard'], selected_df['paragraph_id']))
    missing = [f'{s} {p}' for s,p in required_anchors if (s,p) not in present]
    add('Required anchor paragraphs are captured', len(missing) == 0, f'missing={missing}')
    # Semantic completeness spot checks for previously broken paragraphs.
    def has_text(std, pid, phrase):
        ser = selected_df[(selected_df['standard']==std) & (selected_df['paragraph_id']==pid)]['text']
        return bool(len(ser) and phrase.lower() in ser.iloc[0].lower())
    spot_checks = {
        'S1.20 reporting entity complete': has_text('IFRS S1','20','same reporting entity as the related financial statements'),
        'S1.23 GAAP consistency complete': has_text('IFRS S1','23','requirements of IFRS Accounting Standards or other applicable GAAP'),
        'S1.25 core content complete': has_text('IFRS S1','25','opportunities (see paragraphs 26–27)') and has_text('IFRS S1','25','regulation (see paragraphs 45–53)'),
        'S1.44 includes prioritisation': has_text('IFRS S1','44','whether and how the entity prioritises sustainability-related risks'),
        'S2.29 includes financed emissions': has_text('IFRS S2','29','commercial banking or insurance'),
        'S2.B62 commercial banking complete': has_text('IFRS S2','B62','undrawn loan commitments'),
    }
    failed_spots = [k for k,v in spot_checks.items() if not v]
    add('Semantic spot checks passed', len(failed_spots)==0, f'failed={failed_spots}')
    bank_rows = requirements_df[requirements_df['evidence_tags'].apply(lambda tags: any(t in tags for t in ['commercial_banking','financed_emissions']))]
    add('Banking / financed-emissions requirements are tagged', len(bank_rows) > 0, f'rows={len(bank_rows)}')
    excluded = requirements_df[((requirements_df['standard']=='IFRS S1') & (requirements_df['appendix'].isin(['A','E']))) | ((requirements_df['standard']=='IFRS S2') & (requirements_df['appendix'].isin(['A','C'])))]
    add('Definition and transition appendices are excluded', len(excluded)==0, f'rows={len(excluded)}')
    leaf_count = int(requirements_df['is_leaf_requirement'].sum()) if len(requirements_df) else 0
    add('Leaf requirement rows are identified', leaf_count > 0, f'leaf_rows={leaf_count}')
    footnote_bad = requirements_df[requirements_df['requirement_text'].astype(str).str.contains(r'[.!?]\d{1,2}$', regex=True, na=False)]
    add('No trailing footnote markers remain in requirement text', len(footnote_bad) == 0, f'rows={len(footnote_bad)}')
    has_clean = 'clean_requirement_text' in requirements_df.columns and requirements_df['clean_requirement_text'].notna().all()
    add('Clean generation text is available for every requirement', has_clean, f'column_present={"clean_requirement_text" in requirements_df.columns}')
    if 'generation_bucket' in requirements_df.columns:
        buckets = sorted(requirements_df['generation_bucket'].dropna().unique().tolist())
        add('Rows are separated into generation/context buckets', 'must_disclose_leaf' in buckets and len(buckets) >= 3, f'buckets={buckets}')
    else:
        add('Rows are separated into generation/context buckets', False, 'generation_bucket column missing')
    return pd.DataFrame(rows)


def export_outputs():
    """Run the full pipeline and export all final artefacts."""
    toc_entries, paragraphs_df, selected_df, requirements_df, section_ranges = build_requirements_kb()
    validation_df = validate_outputs(paragraphs_df, selected_df, requirements_df, section_ranges)
    generation_df = build_generation_ready_df(requirements_df)

    req_export = requirements_df.copy()
    selected_export = selected_df.copy()
    generation_export = generation_df.copy()
    for col in ['evidence_tags', 'paragraph_quality_issues', 'requirement_quality_issues']:
        if col in req_export:
            req_export[col] = req_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
        if col in generation_export:
            generation_export[col] = generation_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
    if 'paragraph_quality_issues' in selected_export:
        selected_export['paragraph_quality_issues'] = selected_export['paragraph_quality_issues'].apply(lambda x: json.dumps(x, ensure_ascii=False))

    section_ranges_df = pd.DataFrame(section_ranges)
    toc_df = pd.DataFrame(toc_entries)

    req_export.to_csv(OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline.csv', index=False)
    selected_export.to_csv(OUTPUT_DIR / 'ifrs_s1_s2_selected_paragraphs_deterministic_baseline.csv', index=False)
    section_ranges_df.to_csv(OUTPUT_DIR / 'ifrs_s1_s2_auto_section_ranges_deterministic_baseline.csv', index=False)
    validation_df.to_csv(OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline_validation.csv', index=False)
    toc_df.to_csv(OUTPUT_DIR / 'ifrs_s1_s2_detected_toc_entries_deterministic_baseline.csv', index=False)
    generation_export.to_csv(OUTPUT_DIR / 'ifrs_s1_s2_generation_requirements.csv', index=False)

    requirements_df.to_json(OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline.json', orient='records', indent=2, force_ascii=False)
    generation_df.to_json(OUTPUT_DIR / 'ifrs_s1_s2_generation_requirements.json', orient='records', indent=2, force_ascii=False)
    with open(OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline.jsonl', 'w', encoding='utf-8') as f:
        for row in requirements_df.to_dict(orient='records'):
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    with open(OUTPUT_DIR / 'ifrs_s1_s2_generation_requirements.jsonl', 'w', encoding='utf-8') as f:
        for row in generation_df.to_dict(orient='records'):
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    counts = requirements_df.groupby(['standard','report_section']).size().reset_index(name='requirements')
    leaf_counts = requirements_df[requirements_df['is_leaf_requirement']].groupby(['standard','report_section']).size().reset_index(name='leaf_requirements')
    bucket_counts = requirements_df.groupby(['generation_bucket']).size().reset_index(name='rows')
    generation_counts = generation_df.groupby(['standard','report_section']).size().reset_index(name='generation_rows')
    quality_summary = {
        'min_paragraph_quality_score': float(selected_df['paragraph_quality_score'].min()) if len(selected_df) else None,
        'low_quality_paragraphs': int((selected_df['paragraph_quality_score'] < 0.8).sum()) if len(selected_df) else None,
        'min_requirement_quality_score': float(requirements_df['requirement_quality_score'].min()) if len(requirements_df) else None,
        'low_quality_leaf_requirements': int(((requirements_df['requirement_quality_score'] < 0.8) & (requirements_df['is_leaf_requirement'])).sum()) if len(requirements_df) else None,
    }

    summary = []
    summary.append('# IFRS S1/S2 Automated Requirements KB Audit Summary — Hybrid Azure Version — deterministic baseline\n')
    summary.append('\n## Extraction design\n')
    summary.append('- Section paragraph ranges are not hardcoded. They are derived from each PDF’s official contents page.\n')
    summary.append('- Paragraph text is reconstructed from layout-aware sorted PDF lines, with headers, footers, footnotes and body headings removed before paragraph assembly.\n')
    summary.append('- Appendix guidance is mapped automatically using referenced body paragraphs and strong heading evidence.\n')
    summary.append('- Definition and transition appendices are excluded from the report-generation KB.\n')
    summary.append('- Requirement rows include leaf/container flags so generation agents can prefer actionable leaf requirements.\n')
    summary.append('- Final polish removes trailing PDF footnote markers and adds clean_requirement_text plus generation_bucket.\n')
    summary.append('- A smaller generation-ready export keeps mandatory leaf rows only.\n')
    summary.append('\n## Output counts\n')
    summary.append(f'- Extracted paragraphs: {len(paragraphs_df)}\n')
    summary.append(f'- Selected paragraphs: {len(selected_df)}\n')
    summary.append(f'- Requirement rows: {len(requirements_df)}\n')
    summary.append(f'- Leaf requirement rows: {int(requirements_df["is_leaf_requirement"].sum())}\n')
    summary.append(f'- Generation-ready mandatory leaf rows: {len(generation_df)}\n')
    summary.append('\n## Requirements by section\n\n')
    summary.append(counts.to_markdown(index=False))
    summary.append('\n\n## Leaf requirements by section\n\n')
    summary.append(leaf_counts.to_markdown(index=False))
    summary.append('\n\n## Generation buckets\n\n')
    summary.append(bucket_counts.to_markdown(index=False))
    summary.append('\n\n## Generation-ready mandatory leaf rows by section\n\n')
    summary.append(generation_counts.to_markdown(index=False))
    summary.append('\n\n## Validation checks\n\n')
    summary.append(validation_df.to_markdown(index=False))
    summary.append('\n\n## Text quality summary\n\n')
    for k, v in quality_summary.items():
        summary.append(f'- {k}: {v}\n')
    (OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline_audit_summary.md').write_text(''.join(summary), encoding='utf-8')
    return toc_entries, paragraphs_df, selected_df, requirements_df, section_ranges, validation_df, generation_df


## Step 0 — Input PDF sanity check

This cell verifies that the two source PDFs are available, readable, and have the expected page counts before any extraction starts.

In [2]:
pdf_check_rows = []
for standard, path in PDF_SOURCES.items():
    exists = path.exists()
    pages = None
    file_size_mb = None
    if exists:
        file_size_mb = round(path.stat().st_size / (1024 * 1024), 2)
        with fitz.open(path) as doc:
            pages = len(doc)
    pdf_check_rows.append({
        'standard': standard,
        'path': str(path),
        'exists': exists,
        'pages': pages,
        'file_size_mb': file_size_mb,
    })

pdf_check_df = pd.DataFrame(pdf_check_rows)
display(pdf_check_df)
assert pdf_check_df['exists'].all(), 'One or more source PDFs are missing.'
assert pdf_check_df['pages'].fillna(0).gt(0).all(), 'One or more PDFs could not be read.'

,standard,path,exists,pages,file_size_mb
0,IFRS S1,gen_data\IFRS\ifrs_s1.pdf,True,48,0.29
1,IFRS S2,gen_data\IFRS\ifrs_s2.pdf,True,48,0.29


## Step 1 — Detect the official contents / TOC entries automatically

This cell reads the PDFs' contents pages and detects section starts. These detected starts are later used to compute section ranges, instead of hardcoding IFRS paragraph ranges.

In [3]:
toc_entries = []
for standard, path in PDF_SOURCES.items():
    toc_entries.extend(extract_toc_entries(path, standard))

toc_df = pd.DataFrame(toc_entries)
print('Detected TOC entries:', len(toc_df))
display(toc_df[['standard', 'toc_title', 'start_paragraph', 'end_paragraph', 'toc_page']].head(80))

# Check that all target sections can be inferred from the TOC.
toc_df['mapped_report_section'] = toc_df.apply(lambda r: classify_toc_title(r['toc_title'], r['standard']), axis=1)
toc_section_check = (
    toc_df.dropna(subset=['mapped_report_section'])
          .groupby(['standard', 'mapped_report_section'])
          .size()
          .reset_index(name='toc_entries')
)
display(toc_section_check)

missing_by_standard = {}
for standard in PDF_SOURCES:
    found = set(toc_section_check.loc[toc_section_check['standard'].eq(standard), 'mapped_report_section'])
    missing = sorted(set(TARGET_REPORT_SECTIONS) - found)
    # IFRS S2 has no separate "General Requirements" TOC heading; Objective/Scope are mapped to it.
    if standard == 'IFRS S2' and 'General Requirements' in missing:
        missing.remove('General Requirements')
    missing_by_standard[standard] = missing
print('Missing target TOC mappings:', missing_by_standard)
assert all(len(v) == 0 for v in missing_by_standard.values()), 'Some target sections were not detected from the TOC.'

Detected TOC entries: 31


,standard,toc_title,start_paragraph,end_paragraph,toc_page
0,IFRS S1,OBJECTIVE,1,4.0,4
1,IFRS S1,SCOPE,5,9.0,4
2,IFRS S1,CONCEPTUAL FOUNDATIONS,10,10.0,4
3,IFRS S1,Fair presentation,11,16.0,4
4,IFRS S1,Materiality,17,19.0,4
5,IFRS S1,Reporting entity,20,20.0,4
6,IFRS S1,Connected information,21,24.0,4
7,IFRS S1,CORE CONTENT,25,25.0,4
8,IFRS S1,Governance,26,27.0,4
9,IFRS S1,Strategy,28,42.0,4


,standard,mapped_report_section,toc_entries
0,IFRS S1,General Requirements,18
1,IFRS S1,Governance,1
2,IFRS S1,Metrics and Targets,1
3,IFRS S1,Risk Management,1
4,IFRS S1,Strategy,1
5,IFRS S2,General Requirements,3
6,IFRS S2,Governance,1
7,IFRS S2,Metrics and Targets,1
8,IFRS S2,Risk Management,1
9,IFRS S2,Strategy,1


Missing target TOC mappings: {'IFRS S1': [], 'IFRS S2': []}


## Step 2 — Extract and reconstruct source paragraphs

This is the most important quality step. It reconstructs paragraph-level text from layout-aware PDF lines, then adds paragraph metadata such as ID, page, appendix, heading and sort key.

In [4]:
known_titles = [e['toc_title'] for e in toc_entries] + list(GENERIC_HEADING_STARTS)
paragraphs = []
for standard, path in PDF_SOURCES.items():
    extracted = extract_paragraphs(path, standard, known_titles)
    print(f'{standard}: extracted paragraphs = {len(extracted)}')
    paragraphs.extend(extracted)

paragraphs_df = pd.DataFrame(paragraphs)
paragraphs_df['text'] = paragraphs_df['text'].apply(clean_extracted_text)
fam = paragraphs_df['paragraph_id'].apply(paragraph_family)
paragraphs_df['paragraph_prefix'] = [x[0] for x in fam]
paragraphs_df['paragraph_number'] = [x[1] for x in fam]
paragraphs_df['paragraph_suffix'] = [x[2] for x in fam]
paragraphs_df['paragraph_sort_key'] = paragraphs_df['paragraph_id'].apply(paragraph_sort_key)
paragraphs_df = (
    paragraphs_df
    .sort_values(['standard', 'paragraph_sort_key'])
    .drop(columns=['paragraph_sort_key'])
    .reset_index(drop=True)
)

print('Total extracted paragraphs:', len(paragraphs_df))
display(paragraphs_df.groupby(['standard', 'appendix'], dropna=False).size().reset_index(name='paragraphs'))
display(paragraphs_df[['standard', 'paragraph_id', 'page', 'nearest_heading', 'text']].head(20))

IFRS S1: extracted paragraphs = 188
IFRS S2: extracted paragraphs = 121
Total extracted paragraphs: 309


,standard,appendix,paragraphs
0,IFRS S1,B,59
1,IFRS S1,C,3
2,IFRS S1,D,33
3,IFRS S1,E,7
4,IFRS S1,NaN,86
5,IFRS S2,B,73
6,IFRS S2,C,8
7,IFRS S2,NaN,40


,standard,paragraph_id,page,nearest_heading,text
0,IFRS S1,1,7,Objective,The objective of IFRS S1 General Requirements ...
1,IFRS S1,2,7,Objective,Information about sustainability-related risks...
2,IFRS S1,3,7,Objective,This Standard requires an entity to disclose i...
3,IFRS S1,4,7,Objective,This Standard also prescribes how an entity pr...
4,IFRS S1,5,7,Scope,An entity shall apply this Standard in prepari...
5,IFRS S1,6,7,Scope,Sustainability-related risks and opportunities...
6,IFRS S1,7,7,Scope,Other IFRS Sustainability Disclosure Standards...
7,IFRS S1,7,48,Transition,"Westferry Circus Canary Wharf London E14 4HD, ..."
8,IFRS S1,8,8,Scope,An entity may apply IFRS Sustainability Disclo...
9,IFRS S1,9,8,Scope,This Standard uses terminology suitable for pr...


### Step 2 check — Inspect important paragraph reconstruction

Use this cell to verify that the full paragraph text was reconstructed correctly. Change `CHECK_PARAGRAPHS` whenever you want to inspect another IFRS paragraph.

In [5]:
CHECK_PARAGRAPHS = [
    ('IFRS S1', '20'),   # reporting entity, previously easy to truncate
    ('IFRS S1', '23'),   # connected information, data and assumptions
    ('IFRS S1', '27'),   # governance clauses
    ('IFRS S1', '44'),   # risk management clauses
    ('IFRS S2', '29'),   # climate metrics
    ('IFRS S2', '29A'),  # financed emissions limitation
    ('IFRS S2', 'B62'),  # commercial banking financed emissions
    ('IFRS S2', 'B62A'), # industry and asset-class disaggregation
]

for standard, pid in CHECK_PARAGRAPHS:
    match = paragraphs_df[(paragraphs_df['standard'].eq(standard)) & (paragraphs_df['paragraph_id'].eq(pid))]
    print('=' * 120)
    print(f'{standard} paragraph {pid}')
    if match.empty:
        print('NOT FOUND')
    else:
        row = match.iloc[0]
        print('page:', row['page'], '| heading:', row['nearest_heading'], '| appendix:', row['appendix'])
        print(row['text'])

IFRS S1 paragraph 20
page: 9 | heading: Reporting entity | appendix: nan
An entity’s sustainability-related financial disclosures shall be for the same reporting entity as the related financial statements (see paragraph B38).
IFRS S1 paragraph 23
page: 10 | heading: Connected information | appendix: nan
Data and assumptions used in preparing the sustainability-related financial disclosures shall be consistent—to the extent possible considering the requirements of IFRS Accounting Standards or other applicable GAAP— with the corresponding data and assumptions used in preparing the related financial statements (see paragraph B42).
IFRS S1 paragraph 27
page: 10 | heading: Governance | appendix: nan
To achieve this objective, an entity shall disclose information about: (a) the governance body(s) (which can include a board, committee or equivalent body charged with governance) or individual(s) responsible for oversight of sustainability-related risks and opportunities. Specifically, the enti

## Step 3 — Compute section ranges from detected TOC entries

This cell converts detected contents entries into body paragraph ranges and maps them to the five report sections.

In [6]:
section_ranges = build_body_section_map(toc_entries, paragraphs_df)
section_ranges_df = pd.DataFrame(section_ranges)
print('Computed section ranges:', len(section_ranges_df))
display(section_ranges_df[['standard', 'toc_title', 'report_section', 'start_number', 'end_number', 'toc_page']])

range_check = section_ranges_df.groupby(['standard', 'report_section']).size().reset_index(name='ranges')
display(range_check)
assert set(range_check['report_section']).issubset(set(TARGET_REPORT_SECTIONS)), 'A non-target report section was mapped.'

Computed section ranges: 27


,standard,toc_title,report_section,start_number,end_number,toc_page
0,IFRS S1,OBJECTIVE,General Requirements,1,4,4
1,IFRS S1,SCOPE,General Requirements,5,9,4
2,IFRS S1,CONCEPTUAL FOUNDATIONS,General Requirements,10,10,4
3,IFRS S1,Fair presentation,General Requirements,11,16,4
4,IFRS S1,Materiality,General Requirements,17,19,4
5,IFRS S1,Reporting entity,General Requirements,20,20,4
6,IFRS S1,Connected information,General Requirements,21,24,4
7,IFRS S1,CORE CONTENT,General Requirements,25,25,4
8,IFRS S1,Governance,Governance,26,27,4
9,IFRS S1,Strategy,Strategy,28,42,4


,standard,report_section,ranges
0,IFRS S1,General Requirements,16
1,IFRS S1,Governance,1
2,IFRS S1,Metrics and Targets,1
3,IFRS S1,Risk Management,1
4,IFRS S1,Strategy,1
5,IFRS S2,General Requirements,3
6,IFRS S2,Governance,1
7,IFRS S2,Metrics and Targets,1
8,IFRS S2,Risk Management,1
9,IFRS S2,Strategy,1


## Step 4 — Map body and appendix paragraphs to report sections

Body paragraphs are mapped by automatic TOC ranges. Appendix guidance is mapped by referenced paragraphs and strong heading evidence, then definition and transition appendices are excluded from the report-generation KB.

In [7]:
paragraphs_mapped_df = assign_body_sections(paragraphs_df, section_ranges)
paragraphs_mapped_df = map_appendix_sections(paragraphs_mapped_df)

mapping_counts = (
    paragraphs_mapped_df
    .groupby(['standard', 'report_section', 'mapping_method'], dropna=False)
    .size()
    .reset_index(name='paragraphs')
    .sort_values(['standard', 'report_section', 'mapping_method'])
)
display(mapping_counts)

unmapped_preview = paragraphs_mapped_df[paragraphs_mapped_df['report_section'].isna()][
    ['standard', 'paragraph_id', 'page', 'appendix', 'nearest_heading', 'text']
].head(20)
print('Unmapped paragraphs preview — expected to include definitions, transition, approvals or non-target sections:')
display(unmapped_preview)

,standard,report_section,mapping_method,paragraphs
0,IFRS S1,General Requirements,auto_appendix_reference_or_heading,76
1,IFRS S1,General Requirements,auto_toc_body_range,58
2,IFRS S1,Governance,auto_appendix_reference_or_heading,1
3,IFRS S1,Governance,auto_toc_body_range,2
4,IFRS S1,Metrics and Targets,auto_appendix_reference_or_heading,8
5,IFRS S1,Metrics and Targets,auto_toc_body_range,9
6,IFRS S1,Risk Management,auto_appendix_reference_or_heading,1
7,IFRS S1,Risk Management,auto_toc_body_range,2
8,IFRS S1,Strategy,auto_appendix_reference_or_heading,1
9,IFRS S1,Strategy,auto_toc_body_range,15


Unmapped paragraphs preview — expected to include definitions, transition, approvals or non-target sections:


,standard,paragraph_id,page,appendix,nearest_heading,text
7,IFRS S1,7,48,E,Transition,"Westferry Circus Canary Wharf London E14 4HD, ..."
92,IFRS S1,B6,27,B,Identifying sustainability-related risks and o...,An entity shall use all reasonable and support...
94,IFRS S1,B8,27,B,Reasonable and supportable information,Reasonable and supportable information used by...
96,IFRS S1,B10,28,B,Reasonable and supportable information,An entity need not undertake an exhaustive sea...
97,IFRS S1,B11,28,B,Reasonable and supportable information,On the occurrence of a significant event or si...
98,IFRS S1,B12,28,B,Reasonable and supportable information,"An entity is permitted, but not required, to r..."
115,IFRS S1,B29,31,B,Aggregation and disaggregation,When an entity applies IFRS Sustainability Dis...
116,IFRS S1,B30,32,B,Aggregation and disaggregation,An entity shall not aggregate information if d...
152,IFRS S1,D4,39,D,Introduction,Relevant sustainability-related financial info...
153,IFRS S1,D5,39,D,Introduction,Sustainability-related financial information h...


## Step 5 — Select only target paragraphs and score paragraph quality

This cell creates the paragraph-level source table used for requirement splitting. It also scores paragraph reconstruction quality so suspicious rows can be reviewed before export.

In [8]:
selected_df = paragraphs_mapped_df[paragraphs_mapped_df['report_section'].isin(TARGET_REPORT_SECTIONS)].copy()
selected_df = selected_df[~((selected_df['standard'] == 'IFRS S1') & (selected_df['appendix'].isin(['A', 'E'])))]
selected_df = selected_df[~((selected_df['standard'] == 'IFRS S2') & (selected_df['appendix'].isin(['A', 'C'])))]
selected_df = selected_df[selected_df['text'].str.len() >= 25].copy()

quality_results = selected_df['text'].apply(lambda x: paragraph_quality(x, known_titles))
selected_df['paragraph_quality_score'] = [q[0] for q in quality_results]
selected_df['paragraph_quality_issues'] = [q[1] for q in quality_results]

print('Selected paragraphs:', len(selected_df))
display(selected_df.groupby(['standard', 'report_section']).size().reset_index(name='selected_paragraphs'))

quality_summary = selected_df.groupby(['standard', 'report_section']).agg(
    paragraphs=('paragraph_id', 'count'),
    min_quality=('paragraph_quality_score', 'min'),
    avg_quality=('paragraph_quality_score', 'mean'),
    low_quality=('paragraph_quality_score', lambda s: int((s < 0.8).sum()))
).reset_index()
display(quality_summary)


lowest_quality_sample = selected_df.sort_values(['paragraph_quality_score', 'standard', 'paragraph_id'])[
    ['standard', 'paragraph_id', 'page', 'report_section', 'paragraph_quality_score', 'paragraph_quality_issues', 'text']
].head(20)
print('Lowest-quality paragraph inspection sample, even when validation passes:')
display(lowest_quality_sample)


low_quality_paragraphs = selected_df[selected_df['paragraph_quality_score'] < 0.8][
    ['standard', 'paragraph_id', 'page', 'report_section', 'paragraph_quality_score', 'paragraph_quality_issues', 'text']
]
print('Low-quality selected paragraphs:', len(low_quality_paragraphs))
display(low_quality_paragraphs.head(20))
assert len(low_quality_paragraphs) == 0, 'Low-quality paragraph reconstruction detected. Inspect before exporting.'

Selected paragraphs: 278


,standard,report_section,selected_paragraphs
0,IFRS S1,General Requirements,132
1,IFRS S1,Governance,3
2,IFRS S1,Metrics and Targets,17
3,IFRS S1,Risk Management,3
4,IFRS S1,Strategy,16
5,IFRS S2,General Requirements,4
6,IFRS S2,Governance,3
7,IFRS S2,Metrics and Targets,65
8,IFRS S2,Risk Management,3
9,IFRS S2,Strategy,32


,standard,report_section,paragraphs,min_quality,avg_quality,low_quality
0,IFRS S1,General Requirements,132,1.0,1.000000,0
1,IFRS S1,Governance,3,1.0,1.000000,0
2,IFRS S1,Metrics and Targets,17,1.0,1.000000,0
3,IFRS S1,Risk Management,3,1.0,1.000000,0
4,IFRS S1,Strategy,16,1.0,1.000000,0
5,IFRS S2,General Requirements,4,1.0,1.000000,0
6,IFRS S2,Governance,3,1.0,1.000000,0
7,IFRS S2,Metrics and Targets,65,0.8,0.987692,0
8,IFRS S2,Risk Management,3,1.0,1.000000,0
9,IFRS S2,Strategy,32,1.0,1.000000,0


Lowest-quality paragraph inspection sample, even when validation passes:


,standard,paragraph_id,page,report_section,paragraph_quality_score,paragraph_quality_issues,text
253,IFRS S2,B26,32,Metrics and Targets,0.8,[missing_roman_ii],Paragraph 29(a)(iii) requires an entity to dis...
257,IFRS S2,B30,33,Metrics and Targets,0.8,[missing_roman_iv],Paragraph 29(a)(v) requires an entity to discl...
276,IFRS S2,B49,36,Metrics and Targets,0.8,[heading_leakage],Secondary data for Scope 3 greenhouse gas emis...
281,IFRS S2,B54,37,Metrics and Targets,0.8,[heading_leakage],Verified data might include data that has been...
0,IFRS S1,1,7,General Requirements,1.0,[],The objective of IFRS S1 General Requirements ...
10,IFRS S1,10,8,General Requirements,1.0,[],For sustainability-related financial informati...
11,IFRS S1,11,8,General Requirements,1.0,[],A complete set of sustainability-related finan...
12,IFRS S1,12,8,General Requirements,1.0,[],To identify sustainability-related risks and o...
13,IFRS S1,13,8,General Requirements,1.0,[],Fair presentation requires disclosure of relev...
14,IFRS S1,14,8,General Requirements,1.0,[],Materiality is an entity-specific aspect of re...


Low-quality selected paragraphs: 0


,standard,paragraph_id,page,report_section,paragraph_quality_score,paragraph_quality_issues,text


## Step 6 — Split selected paragraphs into requirement rows

This cell turns paragraph text into requirement-level records. It keeps both container rows and leaf rows. For generation agents, prefer `is_leaf_requirement == True`.

In [9]:
req_rows = []
for row in selected_df.itertuples(index=False):
    clauses = split_requirement_clauses(row.text)
    for clause_index, clause in enumerate(clauses, start=1):
        req_text = clean_extracted_text(clause['requirement_text'])
        clean_req_text = clean_requirement_text_for_generation(req_text)
        tags = infer_tags(' '.join([clean_req_text, str(row.nearest_heading), str(row.official_section_heading)]))
        obligation = infer_obligation(req_text)
        req_quality, req_issues = clause_quality(req_text, clause['is_leaf_requirement'])
        req_rows.append({
            'requirement_id': f"{row.standard.replace(' ', '_')}_{row.paragraph_id}_C{clause_index:02d}",
            'standard': row.standard,
            'paragraph_id': row.paragraph_id,
            'page': int(row.page),
            'report_section': row.report_section,
            'official_section_heading': row.official_section_heading,
            'nearest_pdf_heading': row.nearest_heading,
            'mapping_method': row.mapping_method,
            'appendix': row.appendix,
            'clause_index': clause_index,
            'clause_marker': clause['clause_marker'],
            'clause_level': clause['clause_level'],
            'clause_path': clause['clause_path'],
            'is_leaf_requirement': bool(clause['is_leaf_requirement']),
            'requirement_label': make_label(clean_req_text),
            'requirement_text': req_text,
            'clean_requirement_text': clean_req_text,
            'parent_context': clean_extracted_text(clause['parent_context']),
            'source_paragraph_text': row.text,
            'obligation_type': obligation,
            'mandatory': obligation in {'mandatory', 'prohibition'},
            'generation_bucket': classify_generation_bucket(obligation, clause['is_leaf_requirement'], clean_req_text),
            'evidence_tags': tags,
            'banking_relevance': infer_banking_relevance(row.standard, row.report_section, req_text, tags),
            'paragraph_quality_score': row.paragraph_quality_score,
            'paragraph_quality_issues': row.paragraph_quality_issues,
            'requirement_quality_score': req_quality,
            'requirement_quality_issues': req_issues,
        })

requirements_df = pd.DataFrame(req_rows)
print('Requirement rows:', len(requirements_df))
print('Leaf requirement rows:', int(requirements_df['is_leaf_requirement'].sum()))
display(requirements_df.groupby(['standard', 'report_section', 'is_leaf_requirement']).size().reset_index(name='rows'))
display(requirements_df.groupby(['generation_bucket']).size().reset_index(name='rows'))
display(requirements_df[['requirement_id', 'standard', 'paragraph_id', 'report_section', 'is_leaf_requirement', 'clause_marker', 'generation_bucket', 'clean_requirement_text']].head(30))

Requirement rows: 570
Leaf requirement rows: 532


,standard,report_section,is_leaf_requirement,rows
0,IFRS S1,General Requirements,False,5
1,IFRS S1,General Requirements,True,192
2,IFRS S1,Governance,False,2
3,IFRS S1,Governance,True,9
4,IFRS S1,Metrics and Targets,False,1
5,IFRS S1,Metrics and Targets,True,33
6,IFRS S1,Risk Management,False,1
7,IFRS S1,Risk Management,True,11
8,IFRS S1,Strategy,False,1
9,IFRS S1,Strategy,True,34


,generation_bucket,rows
0,container_context,38
1,must_disclose_leaf,361
2,optional_relief_leaf,56
3,supporting_guidance_leaf,104
4,supporting_objective,11


,requirement_id,standard,paragraph_id,report_section,is_leaf_requirement,clause_marker,generation_bucket,clean_requirement_text
0,IFRS_S1_1_C01,IFRS S1,1,General Requirements,True,NaN,supporting_objective,The objective of IFRS S1 General Requirements ...
1,IFRS_S1_2_C01,IFRS S1,2,General Requirements,True,NaN,supporting_guidance_leaf,Information about sustainability-related risks...
2,IFRS_S1_3_C01,IFRS S1,3,General Requirements,True,NaN,must_disclose_leaf,This Standard requires an entity to disclose i...
3,IFRS_S1_4_C01,IFRS S1,4,General Requirements,True,NaN,supporting_guidance_leaf,This Standard also prescribes how an entity pr...
4,IFRS_S1_5_C01,IFRS S1,5,General Requirements,True,NaN,must_disclose_leaf,An entity shall apply this Standard in prepari...
5,IFRS_S1_6_C01,IFRS S1,6,General Requirements,True,NaN,supporting_guidance_leaf,Sustainability-related risks and opportunities...
6,IFRS_S1_7_C01,IFRS S1,7,General Requirements,True,NaN,must_disclose_leaf,Other IFRS Sustainability Disclosure Standards...
7,IFRS_S1_8_C01,IFRS S1,8,General Requirements,True,NaN,optional_relief_leaf,An entity may apply IFRS Sustainability Disclo...
8,IFRS_S1_9_C01,IFRS S1,9,General Requirements,True,NaN,supporting_guidance_leaf,This Standard uses terminology suitable for pr...
9,IFRS_S1_10_C01,IFRS S1,10,General Requirements,True,NaN,supporting_guidance_leaf,For sustainability-related financial informati...


### Step 6 check — Inspect requirement splitting for any paragraph

Change `CHECK_REQUIREMENT_PARAGRAPH` to verify clause splitting and leaf/container flags for a specific paragraph.

In [10]:
CHECK_REQUIREMENT_PARAGRAPH = ('IFRS S2', 'B62')
mask = requirements_df['standard'].eq(CHECK_REQUIREMENT_PARAGRAPH[0]) & requirements_df['paragraph_id'].eq(CHECK_REQUIREMENT_PARAGRAPH[1])
cols = [
    'requirement_id', 'clause_index', 'clause_marker', 'clause_level', 'clause_path',
    'is_leaf_requirement', 'obligation_type', 'banking_relevance', 'evidence_tags', 'requirement_text'
]
display(requirements_df.loc[mask, cols])

,requirement_id,clause_index,clause_marker,clause_level,clause_path,is_leaf_requirement,obligation_type,banking_relevance,evidence_tags,requirement_text
523,IFRS_S2_B62_C01,1,(a),1,(a),True,mandatory,high,"[ghg_emissions, scope_1, scope_2, scope_3, fin...",An entity that participates in commercial bank...
524,IFRS_S2_B62_C02,2,(b),1,(b),False,mandatory,high,"[commercial_banking, connected_information]",An entity that participates in commercial bank...
525,IFRS_S2_B62_C03,3,(i),2,(b)>(i),True,mandatory,high,"[financial_effects, commercial_banking, connec...",An entity that participates in commercial bank...
526,IFRS_S2_B62_C04,4,(ii),2,(b)>(ii),True,mandatory,high,"[commercial_banking, connected_information]",An entity that participates in commercial bank...
527,IFRS_S2_B62_C05,5,(c),1,(c),False,mandatory,high,"[financed_emissions, commercial_banking]",An entity that participates in commercial bank...
528,IFRS_S2_B62_C06,6,(i),2,(c)>(i),True,mandatory,high,"[financed_emissions, commercial_banking]",An entity that participates in commercial bank...
529,IFRS_S2_B62_C07,7,(ii),2,(c)>(ii),True,mandatory,high,"[financed_emissions, commercial_banking]",An entity that participates in commercial bank...
530,IFRS_S2_B62_C08,8,(iii),2,(c)>(iii),True,mandatory,high,"[financed_emissions, commercial_banking]",An entity that participates in commercial bank...
531,IFRS_S2_B62_C09,9,(d),1,(d),True,mandatory,high,"[financed_emissions, commercial_banking]",An entity that participates in commercial bank...


## Step 6B — Generation-ready mandatory leaf export preview

In [11]:
generation_ready_df = build_generation_ready_df(requirements_df)
print('Generation-ready mandatory leaf rows:', len(generation_ready_df))
display(generation_ready_df.groupby(['standard', 'report_section']).size().reset_index(name='generation_rows'))
display(generation_ready_df.head(25))

# Guidance/context rows remain available in the full KB, but are separated from generation rows.
context_rows = requirements_df[requirements_df['generation_bucket'] != 'must_disclose_leaf'].copy()
print('Separated context / guidance / optional / container rows:', len(context_rows))
display(context_rows.groupby(['generation_bucket']).size().reset_index(name='rows'))


Generation-ready mandatory leaf rows: 361


,standard,report_section,generation_rows
0,IFRS S1,General Requirements,107
1,IFRS S1,Governance,7
2,IFRS S1,Metrics and Targets,28
3,IFRS S1,Risk Management,8
4,IFRS S1,Strategy,23
5,IFRS S2,General Requirements,1
6,IFRS S2,Governance,8
7,IFRS S2,Metrics and Targets,123
8,IFRS S2,Risk Management,9
9,IFRS S2,Strategy,47


,requirement_id,standard,paragraph_id,page,report_section,official_section_heading,nearest_pdf_heading,clean_requirement_text,source_paragraph_text,clause_path,obligation_type,mandatory,evidence_tags,banking_relevance,paragraph_quality_score,requirement_quality_score
0,IFRS_S1_3_C01,IFRS S1,3,7,General Requirements,OBJECTIVE,Objective,This Standard requires an entity to disclose i...,This Standard requires an entity to disclose i...,,mandatory,True,[financial_effects],medium,1.0,1.0
1,IFRS_S1_5_C01,IFRS S1,5,7,General Requirements,SCOPE,Scope,An entity shall apply this Standard in prepari...,An entity shall apply this Standard in prepari...,,mandatory,True,[],general,1.0,1.0
2,IFRS_S1_7_C01,IFRS S1,7,7,General Requirements,SCOPE,Scope,Other IFRS Sustainability Disclosure Standards...,Other IFRS Sustainability Disclosure Standards...,,mandatory,True,[],general,1.0,1.0
3,IFRS_S1_11_C01,IFRS S1,11,8,General Requirements,Fair presentation,Fair presentation,A complete set of sustainability-related finan...,A complete set of sustainability-related finan...,,mandatory,True,[],general,1.0,1.0
4,IFRS_S1_12_C01,IFRS S1,12,8,General Requirements,Fair presentation,Fair presentation,To identify sustainability-related risks and o...,To identify sustainability-related risks and o...,,mandatory,True,[],general,1.0,1.0
5,IFRS_S1_13_C01,IFRS S1,13,8,General Requirements,Fair presentation,Fair presentation,Fair presentation requires disclosure of relev...,Fair presentation requires disclosure of relev...,,mandatory,True,[],general,1.0,1.0
6,IFRS_S1_15_C01,IFRS S1,15,8,General Requirements,Fair presentation,Fair presentation,Fair presentation also requires an entity — (a...,Fair presentation also requires an entity: (a)...,(a),mandatory,True,[],general,1.0,1.0
7,IFRS_S1_15_C02,IFRS S1,15,8,General Requirements,Fair presentation,Fair presentation,Fair presentation also requires an entity — (b...,Fair presentation also requires an entity: (a)...,(b),mandatory,True,[financial_effects],medium,1.0,1.0
8,IFRS_S1_17_C01,IFRS S1,17,9,General Requirements,Materiality,Materiality,An entity shall disclose material information ...,An entity shall disclose material information ...,,mandatory,True,[materiality],general,1.0,1.0
9,IFRS_S1_19_C01,IFRS S1,19,9,General Requirements,Materiality,Materiality,"To identify and disclose material information,...","To identify and disclose material information,...",,mandatory,True,[materiality],general,1.0,1.0


Separated context / guidance / optional / container rows: 209


,generation_bucket,rows
0,container_context,38
1,optional_relief_leaf,56
2,supporting_guidance_leaf,104
3,supporting_objective,11


## Step 7 — Validate final knowledge base quality

This cell runs automated checks for traceability, duplicate IDs, short rows, paragraph reconstruction quality, semantic spot checks, banking tags, and excluded appendices.

In [12]:
validation_df = validate_outputs(paragraphs_mapped_df, selected_df, requirements_df, section_ranges)
display(validation_df)
assert validation_df['passed'].all(), 'One or more validation checks failed. Inspect validation_df before export.'

# Extra check: only actionable generation rows can be filtered cleanly.
leaf_df = requirements_df[requirements_df['is_leaf_requirement']].copy()
print('Leaf rows available for generation:', len(leaf_df))
display(leaf_df.groupby(['standard', 'report_section']).size().reset_index(name='leaf_requirements'))

,check,passed,details
0,Only target report sections are present,True,"actual=['General Requirements', 'Governance', ..."
1,Every requirement has paragraph traceability,True,"{'standard': 0, 'paragraph_id': 0, 'page': 0, ..."
2,Requirement IDs are unique,True,duplicate_ids=0
3,No very short requirement rows,True,rows=0
4,Paragraph reconstruction quality passed,True,low_quality_rows=0
5,Required anchor paragraphs are captured,True,missing=[]
6,Semantic spot checks passed,True,failed=[]
7,Banking / financed-emissions requirements are ...,True,rows=42
8,Definition and transition appendices are excluded,True,rows=0
9,Leaf requirement rows are identified,True,leaf_rows=532


Leaf rows available for generation: 532


,standard,report_section,leaf_requirements
0,IFRS S1,General Requirements,192
1,IFRS S1,Governance,9
2,IFRS S1,Metrics and Targets,33
3,IFRS S1,Risk Management,11
4,IFRS S1,Strategy,34
5,IFRS S2,General Requirements,6
6,IFRS S2,Governance,9
7,IFRS S2,Metrics and Targets,152
8,IFRS S2,Risk Management,10
9,IFRS S2,Strategy,76


## Step 8 — Banking / financed-emissions spot check

This cell verifies that banking-specific IFRS S2 requirements are captured and tagged. This is important for ESG banking report generation.

In [13]:
banking_check = requirements_df[
    requirements_df['banking_relevance'].eq('high') |
    requirements_df['evidence_tags'].apply(lambda tags: bool(set(tags) & {'financed_emissions', 'commercial_banking'}))
].copy()

print('High banking / financed-emissions rows:', len(banking_check))
display(banking_check[[
    'requirement_id', 'standard', 'paragraph_id', 'report_section', 'is_leaf_requirement',
    'banking_relevance', 'evidence_tags', 'requirement_text'
]].head(50))

required_banking_paragraphs = {'29A', '29B', '29C', 'B58', 'B59', 'B60', 'B62', 'B62A'}
found_banking_paragraphs = set(requirements_df.loc[requirements_df['standard'].eq('IFRS S2'), 'paragraph_id'])
missing_banking_paragraphs = sorted(required_banking_paragraphs - found_banking_paragraphs, key=paragraph_sort_key)
print('Missing banking anchor paragraphs:', missing_banking_paragraphs)
assert not missing_banking_paragraphs, 'Some banking / financed-emissions anchor paragraphs were not captured.'

High banking / financed-emissions rows: 47


,requirement_id,standard,paragraph_id,report_section,is_leaf_requirement,banking_relevance,evidence_tags,requirement_text
173,IFRS_S1_B14_C02,IFRS S1,B14,General Requirements,True,high,"[commercial_banking, materiality]",The decisions of primary users relate to provi...
402,IFRS_S2_29_C18,IFRS S2,29,Metrics and Targets,True,high,"[metrics, targets, ghg_emissions, scope_3, fin...",An entity shall disclose information relevant ...
413,IFRS_S2_29A_C01,IFRS S2,29A,Metrics and Targets,True,high,"[metrics, targets, ghg_emissions, scope_3, fin...",In preparing disclosures to meet the requireme...
415,IFRS_S2_29B_C02,IFRS S2,29B,Metrics and Targets,True,high,"[metrics, targets, ghg_emissions, scope_3, fin...",If an entity applies the limitation in paragra...
416,IFRS_S2_29C_C01,IFRS S2,29C,Metrics and Targets,True,high,"[metrics, targets, ghg_emissions, scope_3, fin...",If an entity has included Category 15 greenhou...
491,IFRS_S2_B37_C01,IFRS S2,B37,Metrics and Targets,True,high,"[ghg_emissions, scope_3, financed_emissions, c...",An entity that participates in one or more fin...
513,IFRS_S2_B58_C01,IFRS S2,B58,Metrics and Targets,True,high,"[ghg_emissions, financed_emissions, insurance]",Entities participating in financial activities...
514,IFRS_S2_B59_C01,IFRS S2,B59,Metrics and Targets,True,high,"[ghg_emissions, scope_3, financed_emissions]",Paragraph 29(a)(i) — (3) requires an entity to...
515,IFRS_S2_B59_C02,IFRS S2,B59,Metrics and Targets,True,high,"[financed_emissions, asset_management]",Paragraph 29(a)(i) — (a) asset management (see...
516,IFRS_S2_B59_C03,IFRS S2,B59,Metrics and Targets,True,high,"[financed_emissions, commercial_banking]",Paragraph 29(a)(i) — (b) commercial banking (s...


Missing banking anchor paragraphs: []


## Step 9 — Export outputs and verify files

This cell writes the final CSV, JSON, JSONL, selected-paragraphs file, detected-TOC file, section-ranges file, validation report and audit summary.

In [14]:
# Export CSV-friendly versions with JSON strings for list fields.
generation_ready_df = build_generation_ready_df(requirements_df)

req_export = requirements_df.copy()
selected_export = selected_df.copy()
generation_export = generation_ready_df.copy()
for col in ['evidence_tags', 'paragraph_quality_issues', 'requirement_quality_issues']:
    if col in req_export:
        req_export[col] = req_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
    if col in generation_export:
        generation_export[col] = generation_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
if 'paragraph_quality_issues' in selected_export:
    selected_export['paragraph_quality_issues'] = selected_export['paragraph_quality_issues'].apply(lambda x: json.dumps(x, ensure_ascii=False))

section_ranges_df = pd.DataFrame(section_ranges)
toc_df = pd.DataFrame(toc_entries)

paths = {
    'requirements_csv': OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline.csv',
    'requirements_json': OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline.json',
    'requirements_jsonl': OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline.jsonl',
    'generation_csv': OUTPUT_DIR / 'ifrs_s1_s2_generation_requirements.csv',
    'generation_json': OUTPUT_DIR / 'ifrs_s1_s2_generation_requirements.json',
    'generation_jsonl': OUTPUT_DIR / 'ifrs_s1_s2_generation_requirements.jsonl',
    'selected_paragraphs_csv': OUTPUT_DIR / 'ifrs_s1_s2_selected_paragraphs_deterministic_baseline.csv',
    'section_ranges_csv': OUTPUT_DIR / 'ifrs_s1_s2_auto_section_ranges_deterministic_baseline.csv',
    'detected_toc_csv': OUTPUT_DIR / 'ifrs_s1_s2_detected_toc_entries_deterministic_baseline.csv',
    'validation_csv': OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline_validation.csv',
    'audit_summary_md': OUTPUT_DIR / 'ifrs_s1_s2_requirements_kb_deterministic_baseline_audit_summary.md',
}

req_export.to_csv(paths['requirements_csv'], index=False)
selected_export.to_csv(paths['selected_paragraphs_csv'], index=False)
section_ranges_df.to_csv(paths['section_ranges_csv'], index=False)
toc_df.to_csv(paths['detected_toc_csv'], index=False)
validation_df.to_csv(paths['validation_csv'], index=False)
generation_export.to_csv(paths['generation_csv'], index=False)

requirements_df.to_json(paths['requirements_json'], orient='records', indent=2, force_ascii=False)
generation_ready_df.to_json(paths['generation_json'], orient='records', indent=2, force_ascii=False)
with open(paths['requirements_jsonl'], 'w', encoding='utf-8') as f:
    for row in requirements_df.to_dict(orient='records'):
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
with open(paths['generation_jsonl'], 'w', encoding='utf-8') as f:
    for row in generation_ready_df.to_dict(orient='records'):
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

counts = requirements_df.groupby(['standard','report_section']).size().reset_index(name='requirements')
leaf_counts = requirements_df[requirements_df['is_leaf_requirement']].groupby(['standard','report_section']).size().reset_index(name='leaf_requirements')
bucket_counts = requirements_df.groupby(['generation_bucket']).size().reset_index(name='rows')
generation_counts = generation_ready_df.groupby(['standard','report_section']).size().reset_index(name='generation_rows')
quality_summary = {
    'min_paragraph_quality_score': float(selected_df['paragraph_quality_score'].min()) if len(selected_df) else None,
    'low_quality_paragraphs': int((selected_df['paragraph_quality_score'] < 0.8).sum()) if len(selected_df) else None,
    'min_requirement_quality_score': float(requirements_df['requirement_quality_score'].min()) if len(requirements_df) else None,
    'low_quality_leaf_requirements': int(((requirements_df['requirement_quality_score'] < 0.8) & (requirements_df['is_leaf_requirement'])).sum()) if len(requirements_df) else None,
}

summary = []
summary.append('# IFRS S1/S2 Automated Requirements KB Audit Summary — Hybrid Azure Version — Deterministic Baseline\n')
summary.append('\n## Extraction design\n')
summary.append('- Section paragraph ranges are not hardcoded. They are derived from each PDF’s official contents page.\n')
summary.append('- Paragraph text is reconstructed from layout-aware sorted PDF lines, with headers, footers, footnotes and body headings removed before paragraph assembly.\n')
summary.append('- Appendix guidance is mapped automatically using referenced body paragraphs and strong heading evidence.\n')
summary.append('- Definition and transition appendices are excluded from the report-generation KB.\n')
summary.append('- Requirement rows include leaf/container flags so generation agents can prefer actionable leaf requirements.\n')
summary.append('- Final polish removes trailing PDF footnote markers and adds clean_requirement_text plus generation_bucket.\n')
summary.append('- A smaller generation-ready export keeps mandatory leaf rows only.\n')
summary.append('- The notebook includes verification cells for PDF inputs, TOC detection, paragraph reconstruction, section mapping, quality scoring, splitting, generation filtering, validation and export.\n')
summary.append('\n## Output counts\n')
summary.append(f'- Extracted paragraphs: {len(paragraphs_mapped_df)}\n')
summary.append(f'- Selected paragraphs: {len(selected_df)}\n')
summary.append(f'- Requirement rows: {len(requirements_df)}\n')
summary.append(f'- Leaf requirement rows: {int(requirements_df["is_leaf_requirement"].sum())}\n')
summary.append(f'- Generation-ready mandatory leaf rows: {len(generation_ready_df)}\n')
summary.append('\n## Requirements by section\n\n')
summary.append(counts.to_markdown(index=False))
summary.append('\n\n## Leaf requirements by section\n\n')
summary.append(leaf_counts.to_markdown(index=False))
summary.append('\n\n## Generation buckets\n\n')
summary.append(bucket_counts.to_markdown(index=False))
summary.append('\n\n## Generation-ready mandatory leaf rows by section\n\n')
summary.append(generation_counts.to_markdown(index=False))
summary.append('\n\n## Validation checks\n\n')
summary.append(validation_df.to_markdown(index=False))
summary.append('\n\n## Text quality summary\n\n')
for k, v in quality_summary.items():
    summary.append(f'- {k}: {v}\n')
paths['audit_summary_md'].write_text(''.join(summary), encoding='utf-8')

export_check_df = pd.DataFrame([
    {'output': name, 'path': str(path), 'exists': path.exists(), 'size_kb': round(path.stat().st_size / 1024, 1) if path.exists() else None}
    for name, path in paths.items()
])
display(export_check_df)
assert export_check_df['exists'].all(), 'Some output files were not written successfully.'
print(f'Outputs written to: {OUTPUT_DIR}')


,output,path,exists,size_kb
0,requirements_csv,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,1349.3
1,requirements_json,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,1753.7
2,requirements_jsonl,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,1687.8
3,generation_csv,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,573.8
4,generation_json,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,720.0
5,generation_jsonl,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,695.2
6,selected_paragraphs_csv,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,193.8
7,section_ranges_csv,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,1.8
8,detected_toc_csv,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,1.4
9,validation_csv,gen_data\IFRS\ifrs_requirements_kb_outputs_hyb...,True,1.1


Outputs written to: gen_data\IFRS\ifrs_requirements_kb_outputs_hybrid


## Step 10 — Required Azure OpenAI GPT-5.2 hybrid review layer

This step does **not** replace the deterministic pipeline. It adds a required semantic review layer on top of it.

The notebook fails clearly if Azure GPT-5.2 configuration is missing or invalid. It does not silently skip the LLM review step.

Your existing `.env` format is supported directly:

```env
AZURE_OPENAI_API_KEY=your_shared_resource_key
AZURE_OPENAI_GPT52_DEPLOYMENT_URL=https://your-resource-name.openai.azure.com/openai/deployments/your-gpt-5-2-deployment/chat/completions?api-version=2025-04-01-preview
```

The notebook also supports a cleaner base-endpoint format if you ever want to use it later:

```env
AZURE_OPENAI_API_KEY=your_shared_resource_key
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com
AZURE_OPENAI_API_VERSION=2025-04-01-preview
AZURE_OPENAI_CLASSIFIER_DEPLOYMENT=your-gpt-5-2-deployment-name
```

Review controls:

```env
HYBRID_REVIEW_MODE=smart   # smart | appendix | all
HYBRID_REVIEW_LIMIT=160    # set 0 or empty for no limit
HYBRID_APPLY_LLM_OVERRIDES=true
HYBRID_FAIL_ON_LLM_ERROR=true
HYBRID_SECTION_CONFIDENCE_THRESHOLD=0.82
HYBRID_CLASSIFICATION_CONFIDENCE_THRESHOLD=0.78
```

Backward-compatible full-URL variables accepted by the notebook:

```env
AZURE_OPENAI_GPT52_DEPLOYMENT_URL=...
AZURE_OPENAI_CLASSIFIER_URL=...
AZURE_OPENAI_JUDGE_URL=...
```


In [15]:
# ============================================================
# Required Azure OpenAI hybrid review helpers
# Mirrors the working Azure REST loading style used in your other notebook.
# ============================================================
import os
import json
import re
import urllib.request
import urllib.error
import time
import random

try:
    from dotenv import load_dotenv, find_dotenv
    _env_path = find_dotenv()
    if _env_path:
        load_dotenv(_env_path, override=True)
        print(f"Loaded .env from: {_env_path}")
    else:
        load_dotenv(override=True)
        print("No .env found by find_dotenv(); using existing environment variables.")
except Exception as exc:
    print(f"python-dotenv unavailable or failed to load .env: {exc}")


def _clean_env(value):
    if value is None:
        return None
    value = str(value).strip().strip('"').strip("'")
    return value or None


def _env_bool(name, default=False):
    raw = _clean_env(os.getenv(name))
    if raw is None:
        return default
    return raw.lower() in {"1", "true", "yes", "y", "on"}


def safe_azure_url_preview(url, max_len=120):
    """Show a safe non-secret endpoint preview for debugging."""
    if not url:
        return None
    url = str(url)
    return url if len(url) <= max_len else url[:max_len] + "..."


# ------------------------------------------------------------------
# Configuration loading
# ------------------------------------------------------------------
# This cell intentionally accepts the same full-URL style that works in your
# other notebook. It does NOT rebuild or reorder the deployment URL. Full URL
# env variables win, and are used as-is after quote/space cleanup.
#
# Supported URL variable priority:
# 1) AZURE_OPENAI_GPT52_DEPLOYMENT_URL   <- your current .env name
# 2) AZURE_OPENAI_CLASSIFIER_URL
# 3) AZURE_OPENAI_EXTRACTOR_URL          <- working extraction notebook style
# 4) AZURE_OPENAI_JUDGE_URL              <- governance/judge style
# 5) AZURE_OPENAI_CHAT_URL               <- older fallback style
# ------------------------------------------------------------------

AZURE_OPENAI_SHARED_API_KEY = _clean_env(os.getenv("AZURE_OPENAI_API_KEY"))

_shared_key_fallback = (
    AZURE_OPENAI_SHARED_API_KEY
    or _clean_env(os.getenv("AZURE_OPENAI_CLASSIFIER_API_KEY"))
    or _clean_env(os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY"))
    or _clean_env(os.getenv("AZURE_OPENAI_JUDGE_API_KEY"))
    or _clean_env(os.getenv("AZURE_OPENAI_WRITER_API_KEY"))
    or _clean_env(os.getenv("AZURE_OPENAI_REVISER_API_KEY"))
)

AZURE_OPENAI_CLASSIFIER_API_KEY = (
    _clean_env(os.getenv("AZURE_OPENAI_CLASSIFIER_API_KEY"))
    or _clean_env(os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY"))
    or _clean_env(os.getenv("AZURE_OPENAI_JUDGE_API_KEY"))
    or _shared_key_fallback
)

AZURE_OPENAI_CLASSIFIER_URL = _clean_env(
    os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")
    or os.getenv("AZURE_OPENAI_CLASSIFIER_URL")
    or os.getenv("AZURE_OPENAI_EXTRACTOR_URL")
    or os.getenv("AZURE_OPENAI_JUDGE_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)

# Backward-compatible aliases expected by the next hybrid cells.
AZURE_OPENAI_API_KEY = AZURE_OPENAI_CLASSIFIER_API_KEY


def azure_hybrid_config_available():
    return bool(AZURE_OPENAI_CLASSIFIER_API_KEY and AZURE_OPENAI_CLASSIFIER_URL)


def validate_azure_hybrid_config_required():
    missing = []

    if not AZURE_OPENAI_CLASSIFIER_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY or AZURE_OPENAI_CLASSIFIER_API_KEY / EXTRACTOR_API_KEY / JUDGE_API_KEY")

    if not AZURE_OPENAI_CLASSIFIER_URL:
        missing.append("AZURE_OPENAI_GPT52_DEPLOYMENT_URL or AZURE_OPENAI_CLASSIFIER_URL / EXTRACTOR_URL / JUDGE_URL / CHAT_URL")

    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_SHARED_API_KEY),
            "classifier_key_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_CLASSIFIER_API_KEY"))),
            "extractor_key_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY"))),
            "judge_key_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_JUDGE_API_KEY"))),
            "gpt52_deployment_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL"))),
            "classifier_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_CLASSIFIER_URL"))),
            "extractor_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_EXTRACTOR_URL"))),
            "judge_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_JUDGE_URL"))),
            "chat_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_CHAT_URL"))),
        }
        raise ValueError(
            "Missing Azure GPT-5.2 hybrid configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags, keys are never printed:\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nExpected .env example using your current names:\n"
            + "AZURE_OPENAI_API_KEY=<shared Azure OpenAI resource key>\n"
            + "AZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full GPT-5.2 chat-completions deployment URL>\n\n"
            + "Alternative using the working notebook style:\n"
            + "AZURE_OPENAI_API_KEY=<shared Azure OpenAI resource key>\n"
            + "AZURE_OPENAI_EXTRACTOR_URL=<full GPT-5.2 chat-completions deployment URL>\n"
        )

    if not str(AZURE_OPENAI_CLASSIFIER_URL).startswith("https://"):
        raise ValueError(
            "Azure GPT-5.2 classifier URL must be a full HTTPS deployment URL. "
            f"Current value: {AZURE_OPENAI_CLASSIFIER_URL!r}"
        )

    if "/chat/completions" not in AZURE_OPENAI_CLASSIFIER_URL:
        print("WARNING: The Azure GPT-5.2 URL does not contain '/chat/completions'.")
        print("This cell is using your URL as-is to match the working notebook style.")
        print("Expected full URL format:")
        print("https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=<version>")

    loaded_flags = {
        "api_key_loaded": bool(AZURE_OPENAI_CLASSIFIER_API_KEY),
        "gpt52_deployment_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL"))),
        "classifier_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_CLASSIFIER_URL"))),
        "extractor_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_EXTRACTOR_URL"))),
        "judge_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_JUDGE_URL"))),
        "chat_url_loaded": bool(_clean_env(os.getenv("AZURE_OPENAI_CHAT_URL"))),
    }

    print("Azure GPT-5.2 hybrid configuration loaded.")
    print("Loaded configuration flags, keys are never printed:")
    print(json.dumps(loaded_flags, indent=2))
    print("Classifier endpoint:", safe_azure_url_preview(AZURE_OPENAI_CLASSIFIER_URL))


validate_azure_hybrid_config_required()



def azure_hybrid_config_ok() -> bool:
    """Return True when the required Azure GPT-5.2 configuration validates.

    This helper is used only in the comparison summary. The notebook still calls
    validate_azure_hybrid_config_required() before running reviews, so Azure remains
    required and failures are not silently ignored.
    """
    try:
        validate_azure_hybrid_config_required()
        return True
    except Exception:
        return False


Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Hybrid Azure OpenAI configuration:
{
  "api_key_loaded": true,
  "gpt52_deployment_url_loaded": true,
  "endpoint_loaded": false,
  "api_version": "2024-07-18",
  "classifier_deployment_loaded": false,
  "resolved_classifier_url_loaded": true
}
Classifier endpoint preview: https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.2/chat/completio...


In [16]:

# ============================================================
# GPT-5.2 JSON call and hybrid classification schema
# ============================================================
ALLOWED_OBLIGATION_TYPES = {"mandatory", "relief_or_optional", "guidance", "context_objective", "prohibition"}
ALLOWED_GENERATION_BUCKETS = {"must_disclose_leaf", "optional_relief_leaf", "supporting_guidance_leaf", "supporting_objective", "container_context"}
ALLOWED_BANKING_RELEVANCE = {"general", "medium", "high"}
ALLOWED_EVIDENCE_TAGS = {
    "governance_body", "management_role", "risk_process", "strategy_decision_making",
    "business_model_value_chain", "financial_effects", "materiality", "connected_information",
    "metrics", "targets", "remuneration", "ghg_emissions", "scope_1", "scope_2", "scope_3",
    "financed_emissions", "commercial_banking", "asset_management", "insurance", "carbon_credits",
    "scenario_analysis", "transition_plan", "climate_resilience", "measurement_uncertainty",
}

HYBRID_SYSTEM_PROMPT = """
You are an IFRS S1/S2 disclosure-requirements classifier and validator.
You review one extracted requirement row at a time.

Important rules:
- Do not invent new requirements.
- Do not rewrite the requirement text.
- Use only the provided extracted requirement, source paragraph, heading and metadata.
- Choose report_section only from: General Requirements, Governance, Strategy, Risk Management, Metrics and Targets.
- Appendix A / defined terms must remain excluded from the report-generation KB.
- For body rows mapped by official TOC, preserve the rule-based section unless there is clear evidence of a metadata error.
- For appendix rows, use semantic meaning, referenced body paragraphs and headings.
- If is_leaf_requirement is false, generation_bucket should normally be container_context.
- If the row is a leaf and obligation is mandatory/prohibition, generation_bucket should normally be must_disclose_leaf.
- If the clause marker looks like a year, publication date, citation, paragraph reference or standard title instead of a real list marker, set split_ok=false.

Return one valid JSON object only with exactly these fields:
{
  "report_section": "General Requirements | Governance | Strategy | Risk Management | Metrics and Targets",
  "section_confidence": 0.0,
  "obligation_type": "mandatory | relief_or_optional | guidance | context_objective | prohibition",
  "mandatory": true,
  "generation_bucket": "must_disclose_leaf | optional_relief_leaf | supporting_guidance_leaf | supporting_objective | container_context",
  "evidence_tags": ["tag"],
  "banking_relevance": "general | medium | high",
  "split_ok": true,
  "classification_confidence": 0.0,
  "needs_review": false,
  "reason": "short reason under 35 words"
}
""".strip()



def _extract_json_object(text):
    """Extract the outermost JSON object from model output."""
    text = str(text).strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]
    return text


def _azure_chat_completion_json(
    *,
    url,
    api_key,
    messages,
    max_output_tokens=900,
    timeout=240,
    request_label="Azure GPT-5.2 hybrid classifier",
    max_attempts=5,
):
    """
    Robust Azure REST JSON-mode call.

    This matches the working notebook style:
    - uses the full deployment URL as-is;
    - retries 429/500/502/503/504;
    - honours Retry-After for 429;
    - tries max_completion_tokens first, then max_tokens for gateway compatibility;
    - never prints API keys.
    """
    token_fields = ["max_completion_tokens", "max_tokens"]
    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
            "response_format": {"type": "json_object"},
            "temperature": 0.0,
        }

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={"Content-Type": "application/json", "api-key": api_key},
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    data = json.loads(resp.read().decode("utf-8"))
                    content = data["choices"][0]["message"]["content"]
                    if not content:
                        raise ValueError("Azure returned empty content.")
                    return json.loads(_extract_json_object(content))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {429, 500, 502, 503, 504}
                compatibility_candidate = token_field == "max_completion_tokens" and exc.code in {400, 422, 500}

                if transient and attempt < max_attempts:
                    if exc.code == 429:
                        retry_after_raw = exc.headers.get("Retry-After")
                        try:
                            wait = float(retry_after_raw) if retry_after_raw else 10.0
                        except ValueError:
                            wait = 10.0
                        wait = min(max(wait, 1.0), 60.0)
                        print(
                            f"{request_label}: rate limited (429); retrying "
                            f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                        )
                    else:
                        wait = min(2 ** (attempt - 1) + random.random(), 12)
                        print(
                            f"{request_label}: server error {exc.code}; retrying "
                            f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                        )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support `max_completion_tokens`; "
                        "retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )
                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue
                raise last_error from exc

            except json.JSONDecodeError:
                raise

    raise last_error or RuntimeError(f"{request_label} request failed for an unknown reason.")


def _coerce_float(value, default=None):
    """Safely coerce Azure/LLM numeric confidence values to float.

    Accepts numbers or numeric strings. Returns `default` for missing,
    empty, NaN-like, or non-numeric values so hybrid normalization never
    fails after a successful LLM call.
    """
    if value is None:
        return default
    if isinstance(value, bool):
        return float(value)
    try:
        text = str(value).strip()
        if not text or text.lower() in {"nan", "none", "null", "n/a"}:
            return default
        return float(text)
    except (TypeError, ValueError):
        return default


def _coerce_bool(value, default=None):
    """Safely coerce Azure/LLM boolean-like values to bool."""
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"true", "yes", "y", "1"}:
        return True
    if text in {"false", "no", "n", "0"}:
        return False
    return default

def normalize_hybrid_result(raw):
    """Make model output safe and schema-consistent."""
    if not isinstance(raw, dict):
        raw = {}

    section = raw.get("report_section")
    if section not in TARGET_REPORT_SECTIONS:
        section = None

    obligation = raw.get("obligation_type")
    if obligation not in ALLOWED_OBLIGATION_TYPES:
        obligation = None

    bucket = raw.get("generation_bucket")
    if bucket not in ALLOWED_GENERATION_BUCKETS:
        bucket = None

    tags = raw.get("evidence_tags")
    if not isinstance(tags, list):
        tags = []
    tags = sorted({str(t) for t in tags if str(t) in ALLOWED_EVIDENCE_TAGS})

    banking = raw.get("banking_relevance")
    if banking not in ALLOWED_BANKING_RELEVANCE:
        banking = "general"

    return {
        "hybrid_report_section": section,
        "hybrid_section_confidence": _coerce_float(raw.get("section_confidence")),
        "hybrid_obligation_type": obligation,
        "hybrid_mandatory": _coerce_bool(raw.get("mandatory"), default=None),
        "hybrid_generation_bucket": bucket,
        "hybrid_evidence_tags": tags,
        "hybrid_banking_relevance": banking,
        "hybrid_split_ok": _coerce_bool(raw.get("split_ok", True), default=True),
        "hybrid_classification_confidence": _coerce_float(raw.get("classification_confidence")),
        "hybrid_needs_review": _coerce_bool(raw.get("needs_review", False), default=False),
        "hybrid_reason": normalize_ws(str(raw.get("reason", "")))[:300],
    }


def row_payload_for_hybrid(row):
    """Limit fields sent to the model while preserving traceability and context."""
    def trim(x, n):
        x = normalize_ws(str(x or ""))
        return x[:n]

    return {
        "requirement_id": row.get("requirement_id"),
        "standard": row.get("standard"),
        "paragraph_id": row.get("paragraph_id"),
        "page": row.get("page"),
        "appendix": row.get("appendix"),
        "rule_report_section": row.get("report_section"),
        "official_section_heading": row.get("official_section_heading"),
        "nearest_pdf_heading": row.get("nearest_pdf_heading"),
        "mapping_method": row.get("mapping_method"),
        "clause_marker": row.get("clause_marker"),
        "clause_level": row.get("clause_level"),
        "clause_path": row.get("clause_path"),
        "is_leaf_requirement": bool(row.get("is_leaf_requirement")),
        "rule_obligation_type": row.get("obligation_type"),
        "rule_mandatory": bool(row.get("mandatory")),
        "rule_generation_bucket": row.get("generation_bucket"),
        "rule_evidence_tags": row.get("evidence_tags"),
        "rule_banking_relevance": row.get("banking_relevance"),
        "clean_requirement_text": trim(row.get("clean_requirement_text") or row.get("requirement_text"), 1800),
        "source_paragraph_text": trim(row.get("source_paragraph_text"), 2600),
        "parent_context": trim(row.get("parent_context"), 900),
    }


def classify_row_with_gpt52(row_dict):
    payload = row_payload_for_hybrid(row_dict)
    user_prompt = "Review this extracted IFRS requirement row and return the required JSON object.\n\nROW:\n" + json.dumps(payload, ensure_ascii=False, indent=2)
    raw = _azure_chat_completion_json(
        url=AZURE_OPENAI_CLASSIFIER_URL,
        api_key=AZURE_OPENAI_API_KEY,
        messages=[
            {"role": "system", "content": HYBRID_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=900,
        request_label="Azure GPT-5.2 hybrid classifier",
    )
    return normalize_hybrid_result(raw)

print("Hybrid GPT-5.2 classifier helpers ready")


Hybrid GPT-5.2 classifier helpers ready


In [17]:

# ============================================================
# Select candidate rows for required semantic review
# ============================================================
HYBRID_REVIEW_MODE = (_clean_env(os.getenv("HYBRID_REVIEW_MODE")) or "smart").lower()  # smart | appendix | all
HYBRID_REVIEW_LIMIT_RAW = _clean_env(os.getenv("HYBRID_REVIEW_LIMIT")) or "160"
HYBRID_APPLY_LLM_OVERRIDES = _env_bool("HYBRID_APPLY_LLM_OVERRIDES", default=True)
HYBRID_FAIL_ON_LLM_ERROR = _env_bool("HYBRID_FAIL_ON_LLM_ERROR", default=True)
HYBRID_SECTION_CONFIDENCE_THRESHOLD = float(_clean_env(os.getenv("HYBRID_SECTION_CONFIDENCE_THRESHOLD")) or 0.82)
HYBRID_CLASSIFICATION_CONFIDENCE_THRESHOLD = float(_clean_env(os.getenv("HYBRID_CLASSIFICATION_CONFIDENCE_THRESHOLD")) or 0.78)

ALLOWED_HYBRID_REVIEW_MODES = {"smart", "appendix", "all"}
if HYBRID_REVIEW_MODE not in ALLOWED_HYBRID_REVIEW_MODES:
    raise ValueError(f"HYBRID_REVIEW_MODE must be one of {sorted(ALLOWED_HYBRID_REVIEW_MODES)}, got: {HYBRID_REVIEW_MODE!r}")

try:
    HYBRID_REVIEW_LIMIT = int(HYBRID_REVIEW_LIMIT_RAW)
except Exception:
    HYBRID_REVIEW_LIMIT = 160

if HYBRID_REVIEW_LIMIT <= 0:
    HYBRID_REVIEW_LIMIT = None


def _tags_contain(tags, wanted):
    if isinstance(tags, str):
        try:
            tags = json.loads(tags)
        except Exception:
            tags = [tags]
    if not isinstance(tags, list):
        tags = []
    return any(t in set(tags) for t in wanted)


def select_hybrid_review_rows(requirements_df, mode="smart", limit=160):
    df = requirements_df.copy()

    if mode == "all":
        candidates = df.copy()

    elif mode == "appendix":
        candidates = df[df["appendix"].notna()].copy()

    else:
        # Smart mode: review rows where semantic judgment is most useful.
        appendix_mask = df["appendix"].notna()
        appendix_mapping_mask = df["mapping_method"].astype(str).str.contains("appendix", case=False, na=False)
        lower_quality_mask = (df["paragraph_quality_score"] < 1.0) | (df["requirement_quality_score"] < 1.0)
        numeric_marker_issue_mask = df["clause_marker"].astype(str).str.match(r"^\(\d{2,}\)$", na=False)
        banking_mask = df["evidence_tags"].apply(lambda x: _tags_contain(x, {"financed_emissions", "commercial_banking", "asset_management", "insurance"}))
        optional_or_guidance_mask = df["generation_bucket"].isin(["optional_relief_leaf", "supporting_guidance_leaf"])

        candidates = df[
            appendix_mask |
            appendix_mapping_mask |
            lower_quality_mask |
            numeric_marker_issue_mask |
            banking_mask |
            optional_or_guidance_mask
        ].copy()

    # Always exclude definition / transition appendices from LLM review and generation KB.
    excluded_appendix = (
        ((candidates["standard"] == "IFRS S1") & candidates["appendix"].isin(["A", "E"])) |
        ((candidates["standard"] == "IFRS S2") & candidates["appendix"].isin(["A", "C"]))
    )
    candidates = candidates[~excluded_appendix].copy()

    # Prioritise most impactful rows first.
    candidates["_priority"] = 0
    candidates.loc[candidates["appendix"].notna(), "_priority"] += 50
    candidates.loc[candidates["mapping_method"].astype(str).str.contains("appendix", case=False, na=False), "_priority"] += 30
    candidates.loc[candidates["generation_bucket"].eq("must_disclose_leaf"), "_priority"] += 20
    candidates.loc[candidates["evidence_tags"].apply(lambda x: _tags_contain(x, {"financed_emissions", "commercial_banking"})), "_priority"] += 15
    candidates.loc[(candidates["paragraph_quality_score"] < 1.0) | (candidates["requirement_quality_score"] < 1.0), "_priority"] += 10

    candidates = candidates.sort_values(["_priority", "standard", "paragraph_id", "clause_index"], ascending=[False, True, True, True])
    if limit is not None:
        candidates = candidates.head(limit)
    return candidates.drop(columns=["_priority"], errors="ignore").reset_index(drop=True)


hybrid_candidate_rows = select_hybrid_review_rows(requirements_df, mode=HYBRID_REVIEW_MODE, limit=HYBRID_REVIEW_LIMIT)
print(f"Hybrid review mode: {HYBRID_REVIEW_MODE}")
print(f"Hybrid apply LLM overrides: {HYBRID_APPLY_LLM_OVERRIDES}")
print(f"Hybrid fail on LLM error: {HYBRID_FAIL_ON_LLM_ERROR}")
print(f"Candidate rows selected for GPT-5.2 review: {len(hybrid_candidate_rows)} / {len(requirements_df)}")

if len(hybrid_candidate_rows) == 0:
    raise ValueError(
        "Azure GPT-5.2 hybrid review is required, but zero candidate rows were selected. "
        "Use HYBRID_REVIEW_MODE=all or check the deterministic extraction output."
    )

if len(hybrid_candidate_rows):
    display(hybrid_candidate_rows[[
        "requirement_id", "standard", "paragraph_id", "appendix", "report_section",
        "mapping_method", "generation_bucket", "evidence_tags", "clean_requirement_text"
    ]].head(10))


Hybrid review mode: smart
Hybrid apply LLM overrides: True
Hybrid fail on LLM error: True
Candidate rows selected for GPT-5.2 review: 160 / 570


,requirement_id,standard,paragraph_id,appendix,report_section,mapping_method,generation_bucket,evidence_tags,clean_requirement_text
0,IFRS_S2_B37_C01,IFRS S2,B37,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[ghg_emissions, scope_3, financed_emissions, c...",An entity that participates in one or more fin...
1,IFRS_S2_B59_C01,IFRS S2,B59,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[ghg_emissions, scope_3, financed_emissions]",Paragraph 29(a)(i) — (3) requires an entity to...
2,IFRS_S2_B60_C01,IFRS S2,B60,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[ghg_emissions, financed_emissions]",An entity shall apply the requirements for dis...
3,IFRS_S2_B61_C01,IFRS S2,B61,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[ghg_emissions, scope_1, scope_2, scope_3, fin...",An entity that participates in asset managemen...
4,IFRS_S2_B61_C02,IFRS S2,B61,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[financed_emissions, asset_management, connect...",An entity that participates in asset managemen...
5,IFRS_S2_B61_C03,IFRS S2,B61,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[financed_emissions, asset_management]",An entity that participates in asset managemen...
6,IFRS_S2_B61_C04,IFRS S2,B61,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[financed_emissions, asset_management]",An entity that participates in asset managemen...
7,IFRS_S2_B62_C01,IFRS S2,B62,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[ghg_emissions, scope_1, scope_2, scope_3, fin...",An entity that participates in commercial bank...
8,IFRS_S2_B62_C03,IFRS S2,B62,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[financial_effects, commercial_banking, connec...",An entity that participates in commercial bank...
9,IFRS_S2_B62_C04,IFRS S2,B62,B,Metrics and Targets,auto_appendix_reference_or_heading,must_disclose_leaf,"[commercial_banking, connected_information]",An entity that participates in commercial bank...


In [18]:

# ============================================================
# Run required GPT-5.2 review and apply conservative high-confidence overrides
# ============================================================


def run_hybrid_reviews(candidates_df):
    review_rows = []
    for i, row in candidates_df.iterrows():
        rid = row["requirement_id"]
        print(f"[{i+1}/{len(candidates_df)}] Reviewing {rid}...")
        try:
            result = classify_row_with_gpt52(row.to_dict())
            result["requirement_id"] = rid
            result["hybrid_error"] = ""
        except Exception as exc:
            if HYBRID_FAIL_ON_LLM_ERROR:
                raise RuntimeError(
                    f"Required Azure GPT-5.2 review failed for {rid}. "
                    "Fix the Azure endpoint/configuration or set HYBRID_FAIL_ON_LLM_ERROR=false only if you intentionally want a manual-review row."
                ) from exc
            result = {
                "requirement_id": rid,
                "hybrid_report_section": None,
                "hybrid_section_confidence": 0.0,
                "hybrid_obligation_type": None,
                "hybrid_mandatory": None,
                "hybrid_generation_bucket": None,
                "hybrid_evidence_tags": [],
                "hybrid_banking_relevance": None,
                "hybrid_split_ok": False,
                "hybrid_classification_confidence": 0.0,
                "hybrid_needs_review": True,
                "hybrid_reason": "LLM call failed; manual review needed.",
                "hybrid_error": str(exc)[:1000],
            }
        review_rows.append(result)
    return pd.DataFrame(review_rows)


def apply_hybrid_reviews(requirements_df, reviews_df):
    out = requirements_df.copy()

    # Preserve deterministic outputs.
    for col in ["report_section", "obligation_type", "mandatory", "generation_bucket", "evidence_tags", "banking_relevance"]:
        out[f"rule_{col}"] = out[col]

    out["final_report_section"] = out["report_section"]
    out["final_obligation_type"] = out["obligation_type"]
    out["final_mandatory"] = out["mandatory"]
    out["final_generation_bucket"] = out["generation_bucket"]
    out["final_evidence_tags"] = out["evidence_tags"]
    out["final_banking_relevance"] = out["banking_relevance"]
    out["hybrid_reviewed"] = False
    out["hybrid_split_ok"] = True
    out["hybrid_needs_review"] = False
    out["hybrid_reason"] = ""
    out["hybrid_section_confidence"] = None
    out["hybrid_classification_confidence"] = None

    if reviews_df is None or len(reviews_df) == 0:
        return out

    review_map = {r["requirement_id"]: r for r in reviews_df.to_dict(orient="records")}

    for idx, row in out.iterrows():
        rid = row["requirement_id"]
        r = review_map.get(rid)
        if not r:
            continue

        out.at[idx, "hybrid_reviewed"] = True
        out.at[idx, "hybrid_split_ok"] = bool(r.get("hybrid_split_ok", True))
        out.at[idx, "hybrid_needs_review"] = bool(r.get("hybrid_needs_review", False)) or not bool(r.get("hybrid_split_ok", True))
        out.at[idx, "hybrid_reason"] = r.get("hybrid_reason", "")
        out.at[idx, "hybrid_section_confidence"] = r.get("hybrid_section_confidence")
        out.at[idx, "hybrid_classification_confidence"] = r.get("hybrid_classification_confidence")

        # Keep body TOC section mapping conservative. Only appendix/reference rows can be overridden semantically.
        can_override_section = (
            HYBRID_APPLY_LLM_OVERRIDES and
            pd.notna(row.get("appendix")) and
            r.get("hybrid_report_section") in TARGET_REPORT_SECTIONS and
            float(r.get("hybrid_section_confidence") or 0) >= HYBRID_SECTION_CONFIDENCE_THRESHOLD
        )
        if can_override_section:
            out.at[idx, "final_report_section"] = r["hybrid_report_section"]

        can_override_classification = (
            HYBRID_APPLY_LLM_OVERRIDES and
            bool(r.get("hybrid_split_ok", True)) and
            float(r.get("hybrid_classification_confidence") or 0) >= HYBRID_CLASSIFICATION_CONFIDENCE_THRESHOLD
        )
        if can_override_classification:
            if r.get("hybrid_obligation_type") in ALLOWED_OBLIGATION_TYPES:
                out.at[idx, "final_obligation_type"] = r["hybrid_obligation_type"]
            if r.get("hybrid_mandatory") is not None:
                out.at[idx, "final_mandatory"] = bool(r["hybrid_mandatory"])
            if r.get("hybrid_generation_bucket") in ALLOWED_GENERATION_BUCKETS:
                out.at[idx, "final_generation_bucket"] = r["hybrid_generation_bucket"]
            if isinstance(r.get("hybrid_evidence_tags"), list):
                # Union keeps deterministic traceability and adds semantic tags.
                merged_tags = sorted(set(row.get("evidence_tags") or []) | set(r["hybrid_evidence_tags"]))
                out.at[idx, "final_evidence_tags"] = merged_tags
            if r.get("hybrid_banking_relevance") in ALLOWED_BANKING_RELEVANCE:
                out.at[idx, "final_banking_relevance"] = r["hybrid_banking_relevance"]

    return out


validate_azure_hybrid_config_required()

# Save/reuse review results so a later notebook error does not force you to pay/wait
# for the same 160 GPT-5.2 calls again. Delete this file if you want a fresh review.
HYBRID_REVIEW_CACHE_PATH = OUTPUT_DIR / "ifrs_s1_s2_hybrid_review_results_cache.csv"

if HYBRID_REVIEW_CACHE_PATH.exists():
    cached_reviews_df = pd.read_csv(HYBRID_REVIEW_CACHE_PATH)
    expected_ids = set(hybrid_candidate_rows["requirement_id"].astype(str))
    cached_ids = set(cached_reviews_df.get("requirement_id", pd.Series(dtype=str)).astype(str))
    if expected_ids.issubset(cached_ids):
        print(f"Reusing cached GPT-5.2 hybrid reviews from: {HYBRID_REVIEW_CACHE_PATH}")
        hybrid_reviews_df = cached_reviews_df[cached_reviews_df["requirement_id"].astype(str).isin(expected_ids)].copy()
    else:
        print("Existing hybrid review cache does not match current candidates; running fresh GPT-5.2 review...")
        hybrid_reviews_df = run_hybrid_reviews(hybrid_candidate_rows)
        to_csv_friendly(hybrid_reviews_df).to_csv(HYBRID_REVIEW_CACHE_PATH, index=False)
else:
    print("Running required Azure GPT-5.2 hybrid review...")
    hybrid_reviews_df = run_hybrid_reviews(hybrid_candidate_rows)
    to_csv_friendly(hybrid_reviews_df).to_csv(HYBRID_REVIEW_CACHE_PATH, index=False)

if len(hybrid_reviews_df) != len(hybrid_candidate_rows):
    raise RuntimeError(
        f"Required Azure GPT-5.2 review did not review all selected rows: "
        f"{len(hybrid_reviews_df)} reviewed / {len(hybrid_candidate_rows)} selected."
    )

requirements_hybrid_df = apply_hybrid_reviews(requirements_df, hybrid_reviews_df)

# Build a generation-ready dataframe from final_* decisions, while preserving original columns too.
_generation_source = requirements_hybrid_df.copy()
_generation_source["report_section"] = _generation_source["final_report_section"]
_generation_source["obligation_type"] = _generation_source["final_obligation_type"]
_generation_source["mandatory"] = _generation_source["final_mandatory"]
_generation_source["generation_bucket"] = _generation_source["final_generation_bucket"]
_generation_source["evidence_tags"] = _generation_source["final_evidence_tags"]
_generation_source["banking_relevance"] = _generation_source["final_banking_relevance"]

generation_ready_hybrid_df = build_generation_ready_df(_generation_source)

comparison_summary = {
    "azure_hybrid_required": True,
    "azure_hybrid_config_ok": True,
    "hybrid_review_mode": HYBRID_REVIEW_MODE,
    "hybrid_candidate_rows": int(len(hybrid_candidate_rows)),
    "hybrid_reviewed_rows": int(len(hybrid_reviews_df)),
    "deterministic_requirement_rows": int(len(requirements_df)),
    "hybrid_requirement_rows": int(len(requirements_hybrid_df)),
    "deterministic_generation_rows": int(len(generation_ready_df)),
    "hybrid_generation_rows": int(len(generation_ready_hybrid_df)),
    "section_changes_applied": int((requirements_hybrid_df["final_report_section"] != requirements_hybrid_df["rule_report_section"]).sum()),
    "obligation_changes_applied": int((requirements_hybrid_df["final_obligation_type"] != requirements_hybrid_df["rule_obligation_type"]).sum()),
    "bucket_changes_applied": int((requirements_hybrid_df["final_generation_bucket"] != requirements_hybrid_df["rule_generation_bucket"]).sum()),
    "split_issues_flagged": int((requirements_hybrid_df["hybrid_split_ok"] == False).sum()),
    "manual_review_rows": int((requirements_hybrid_df["hybrid_needs_review"] == True).sum()),
}

print(json.dumps(comparison_summary, indent=2))

if len(hybrid_reviews_df):
    display(hybrid_reviews_df.head(10))


Running required Azure GPT-5.2 hybrid review...
[1/160] Reviewing IFRS_S2_B37_C01...


RuntimeError: Required Azure GPT-5.2 review failed for IFRS_S2_B37_C01. Fix the Azure endpoint/configuration or set HYBRID_FAIL_ON_LLM_ERROR=false only if you intentionally want a manual-review row.

In [ ]:

# ============================================================
# Export hybrid outputs and comparison artefacts
# ============================================================
HYBRID_OUTPUT_DIR = OUTPUT_DIR
HYBRID_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def to_csv_friendly(df):
    df = df.copy()
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, (list, dict))).any():
            df[col] = df[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)
    return df

requirements_hybrid_export = to_csv_friendly(requirements_hybrid_df)
generation_ready_hybrid_export = to_csv_friendly(generation_ready_hybrid_df)
hybrid_reviews_export = to_csv_friendly(hybrid_reviews_df) if len(hybrid_reviews_df) else pd.DataFrame()

requirements_hybrid_export.to_csv(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_requirements_kb_hybrid.csv", index=False)
generation_ready_hybrid_export.to_csv(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_generation_requirements_hybrid.csv", index=False)

requirements_hybrid_df.to_json(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_requirements_kb_hybrid.json", orient="records", indent=2, force_ascii=False)
generation_ready_hybrid_df.to_json(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_generation_requirements_hybrid.json", orient="records", indent=2, force_ascii=False)

with open(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_requirements_kb_hybrid.jsonl", "w", encoding="utf-8") as f:
    for row in requirements_hybrid_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_generation_requirements_hybrid.jsonl", "w", encoding="utf-8") as f:
    for row in generation_ready_hybrid_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

if len(hybrid_reviews_export):
    hybrid_reviews_export.to_csv(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_hybrid_review_results.csv", index=False)
else:
    pd.DataFrame(columns=["requirement_id", "hybrid_reason"]).to_csv(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_hybrid_review_results.csv", index=False)

pd.DataFrame([comparison_summary]).to_csv(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_hybrid_comparison_summary.csv", index=False)

manual_review_df = requirements_hybrid_df[requirements_hybrid_df["hybrid_needs_review"] == True].copy()
to_csv_friendly(manual_review_df).to_csv(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_hybrid_manual_review_queue.csv", index=False)

hybrid_counts = requirements_hybrid_df.groupby(["standard", "final_report_section"]).size().reset_index(name="requirements")
hybrid_generation_counts = generation_ready_hybrid_df.groupby(["standard", "report_section"]).size().reset_index(name="generation_rows")

summary_lines = []
summary_lines.append("# IFRS S1/S2 Hybrid Azure OpenAI Review Summary\n")
summary_lines.append("\n## Design\n")
summary_lines.append("- Deterministic extraction remains the auditable baseline.\n")
summary_lines.append("- Appendix A / defined terms remains excluded from the report-generation KB.\n")
summary_lines.append("- Azure GPT-5.2 is a required semantic review layer; missing config or failed calls stop the notebook by default.\n")
summary_lines.append("- Original rule-based columns are preserved as `rule_*`; hybrid-applied decisions are stored as `final_*`.\n")
summary_lines.append("- Body paragraph section mapping from official TOC is preserved conservatively; section overrides are allowed only for appendix rows above the confidence threshold.\n")
summary_lines.append("\n## Comparison summary\n")
summary_lines.append(pd.DataFrame([comparison_summary]).to_markdown(index=False))
summary_lines.append("\n\n## Hybrid requirements by final section\n")
summary_lines.append(hybrid_counts.to_markdown(index=False))
summary_lines.append("\n\n## Hybrid generation-ready rows by section\n")
summary_lines.append(hybrid_generation_counts.to_markdown(index=False) if len(hybrid_generation_counts) else "No generation-ready rows.")
summary_lines.append("\n\n## Manual review queue\n")
summary_lines.append(f"Rows flagged for manual review: {len(manual_review_df)}\n")

with open(HYBRID_OUTPUT_DIR / "ifrs_s1_s2_hybrid_audit_summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("Hybrid outputs exported to:", HYBRID_OUTPUT_DIR.resolve())
for name in [
    "ifrs_s1_s2_requirements_kb_hybrid.csv",
    "ifrs_s1_s2_generation_requirements_hybrid.csv",
    "ifrs_s1_s2_hybrid_review_results.csv",
    "ifrs_s1_s2_hybrid_comparison_summary.csv",
    "ifrs_s1_s2_hybrid_manual_review_queue.csv",
    "ifrs_s1_s2_hybrid_audit_summary.md",
]:
    path = HYBRID_OUTPUT_DIR / name
    print(f"- {name}: {path.stat().st_size:,} bytes")


## Step 11 — How to decide whether the hybrid version is better

After running the required Azure GPT-5.2 layer, inspect these files:

- `ifrs_s1_s2_hybrid_comparison_summary.csv` — shows how many rows changed and how many were flagged.
- `ifrs_s1_s2_hybrid_manual_review_queue.csv` — most important file to manually inspect.
- `ifrs_s1_s2_hybrid_review_results.csv` — raw GPT-5.2 classification outputs.
- `ifrs_s1_s2_generation_requirements_hybrid.csv` — generation-ready version after hybrid decisions.

The hybrid version is better only if it improves ambiguous appendix mappings or classification quality **without** losing mandatory requirements, traceability, or section coverage.


## Optional — Quick reusable inspection helpers

Run this cell whenever you want to manually inspect one paragraph or one requirement by ID.

In [ ]:
def inspect_paragraph(standard, paragraph_id):
    rows = selected_df[(selected_df['standard'].eq(standard)) & (selected_df['paragraph_id'].eq(str(paragraph_id)))]
    if rows.empty:
        rows = paragraphs_mapped_df[(paragraphs_mapped_df['standard'].eq(standard)) & (paragraphs_mapped_df['paragraph_id'].eq(str(paragraph_id)))]
    if rows.empty:
        print(f'No paragraph found for {standard} {paragraph_id}')
        return
    row = rows.iloc[0]
    print(f"{standard} paragraph {paragraph_id} | page {row['page']} | section: {row.get('report_section')} | heading: {row.get('nearest_heading')}")
    print('-' * 120)
    print(row['text'])


def inspect_requirements(standard, paragraph_id, leaf_only=False):
    rows = requirements_df[(requirements_df['standard'].eq(standard)) & (requirements_df['paragraph_id'].eq(str(paragraph_id)))]
    if leaf_only:
        rows = rows[rows['is_leaf_requirement']]
    cols = ['requirement_id', 'clause_marker', 'clause_level', 'is_leaf_requirement', 'obligation_type', 'generation_bucket', 'banking_relevance', 'evidence_tags', 'clean_requirement_text']
    display(rows[cols])

# Examples:
inspect_paragraph('IFRS S2', 'B62')
inspect_requirements('IFRS S2', 'B62', leaf_only=True)


def inspect_generation_requirements(section=None, standard=None, banking_only=False, limit=20):
    rows = generation_ready_df.copy()
    if section:
        rows = rows[rows['report_section'].eq(section)]
    if standard:
        rows = rows[rows['standard'].eq(standard)]
    if banking_only:
        rows = rows[rows['banking_relevance'].isin(['high', 'medium'])]
    display(rows.head(limit))


## Recommended downstream use

For report-generation agents, start from `ifrs_s1_s2_generation_requirements.csv` or `generation_ready_df`. It contains only mandatory leaf requirements and uses `clean_requirement_text` as the main text field.

Use the full KB when you need audit traceability or context rows. The `generation_bucket` column separates:

- `must_disclose_leaf` — primary rows for generation;
- `optional_relief_leaf` — optional/relief rows;
- `supporting_guidance_leaf` — useful guidance/context;
- `supporting_objective` — section objectives;
- `container_context` — parent clauses that introduce subclauses.

For auditing, use the selected paragraph export and the validation CSV to verify traceability back to the source PDF paragraph.